# Phone camera / IMU calibration

Researcher setup per physical phone and native camera configuration, not per participant. This notebook processes timestamped ChArUco capture from GaitTrace. It never enables distance by fitting to the claimed walking distance. Geometry candidates still need an odometry implementation and independent route validation.

Run with the project Python environment containing OpenCV ArUco, NumPy and SciPy. Generated images, captures and reports remain local/ignored. Actual user capture calibration-1791297358282 was inspected and processed on 6 October 2026. Sensor acquisition and lens fitting are supported; complete camera/IMU alignment remains rejected. Numerical tests remain separate from the actual capture results.

In [5]:
from pathlib import Path
import sys, subprocess, json
ROOT = Path.cwd() if (Path.cwd()/"android").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/"android/research"))
from camera_calibration import write_board, process_bundle
OUTPUT = ROOT/"android/dist/camera-calibration"
print(write_board(OUTPUT))
BUNDLE = Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791297358282.zip")


C:\Users\frank\Documents\MR-ICT Review Paper\android\dist\camera-calibration\print-board.html


## Researcher capture

Use either the printed board or a second display with 20 mm squares verified. Keep the board/display fixed on a table or stand throughout capture; move only the recording phone. Keep the same holder window in front of the lens, with the front camera able to see the board. In Researcher tools > Phone calibration, tap Capture calibration. Follow the brief still/move cues: move closer/further, side-to-side and tilt around all three axes while keeping the board visible. The 40-second capture stops automatically. Export its ZIP. This is a researcher bench routine, not a patient walking test.

The first five seconds are quiet; remaining motion establishes camera geometry and camera/IMU alignment. Camera frames, exposure/skew results and all three native sensor streams preserve actual timestamps and sensor units. Acquisition requests do not guarantee the delivered sampling rate. Front-camera optical obstructions, blur, timing gaps and insufficient motion can make a capture unusable.

In [25]:
check = subprocess.run([sys.executable,"-m","unittest","discover","-s",str(ROOT/"android/research"),"-p","test_*.py","-v"],capture_output=True,text=True)
print(check.stderr)
assert check.returncode == 0

test_acceleration_geometry_recovers_known_coefficients (test_camera_calibration.CalibrationTests.test_acceleration_geometry_recovers_known_coefficients) ... ok
test_hardware_time_uses_feature_row_and_exposure (test_camera_calibration.CalibrationTests.test_hardware_time_uses_feature_row_and_exposure) ... ok
test_interrupted_capture_never_activates_distance (test_camera_calibration.CalibrationTests.test_interrupted_capture_never_activates_distance) ... ok
test_printed_board_detects_expected_corners (test_camera_calibration.CalibrationTests.test_printed_board_detects_expected_corners) ... ok
test_projection_shape_aliases_compare_corresponding_points (test_camera_calibration.CalibrationTests.test_projection_shape_aliases_compare_corresponding_points) ... ok
test_recovers_rotation_bias_and_offset_from_synthetic_motion (test_camera_calibration.CalibrationTests.test_recovers_rotation_bias_and_offset_from_synthetic_motion) ... ok
test_stillness_is_not_calibration (test_camera_calibration.Calib

## Process the exported calibration ZIP

Set BUNDLE to the actual exported file. Missing captures are left pending, not passed. Checks fit lens intrinsics on training views and evaluate held-out views, estimate rotation/time offset using visual angular motion and the gyroscope, then screen lever arm/accelerometer consistency. Numerical tolerances are engineering admission thresholds, not established clinical accuracy criteria. Interleaved held-out samples are an internal check within the same capture, not independent external validation. Only accepted geometry reports may be imported on the originating phone.

## Actual supplied capture: 6 October 2026

Native ZIP quality and visual board detection; actual processing, not artifact replay. Original capture stays unchanged outside the repository. Images/reports remain local, not embedded in source.


In [10]:
import hashlib, zipfile, numpy as np, cv2
from PIL import Image, ImageDraw
from camera_calibration import read_bundle, board
REVIEW = OUTPUT/BUNDLE.stem
REVIEW.mkdir(parents=True,exist_ok=True)
z, manifest, frames, imu, capture_results = read_bundle(BUNDLE)
summary = {"sourceSha256":hashlib.sha256(BUNDLE.read_bytes()).hexdigest(), "captureId":manifest["captureId"], "status":manifest["status"], "device":manifest["device"], "nativeDurationSeconds":(manifest["endedBootNs"]-manifest["startedBootNs"])/1e9, "cameraDimensions":[manifest["width"],manifest["height"]], "timestampSource":manifest["timestampSource"], "streams":{}}
for sensor in ["accelerometer","gyroscope","magnetometer"]:
    rows=[v for v in imu if v["sensor"]==sensor]
    times=np.array([int(v["timestampNs"]) for v in rows],dtype=np.int64)
    gaps=np.diff(times)/1e6
    xyz=np.array([[v[k] for k in ["x","y","z"]] for v in rows])
    summary["streams"][sensor]={"readings":len(rows), "observedHz":(len(rows)-1)*1e9/(times[-1]-times[0]), "maxGapMs":float(gaps.max()), "p99GapMs":float(np.percentile(gaps,99)), "nonmonotonic":int((gaps<=0).sum()), "finite":bool(np.isfinite(xyz).all()), "units":manifest["units"][sensor]}
    if sensor=="gyroscope":
        start=manifest["startedBootNs"]
        summary["quietFirstFiveSecondsGyroRmsRadS"]=float(np.sqrt(np.mean(xyz[(times-start)<5e9]**2)))
        summary["gyroAxisStdRadS"]=np.std(xyz,axis=0).tolist()
times=np.array([int(v["timestampNs"]) for v in frames],dtype=np.int64)
summary["camera"]={"frames":len(frames), "savedFrameHz":(len(frames)-1)*1e9/(times[-1]-times[0]), "maxGapMs":float((np.diff(times)/1e6).max()), "resultMatches":sum(int(v["timestampNs"]) in capture_results for v in frames)}
canvas=Image.new("RGB",(1280,1120),"white");draw=ImageDraw.Draw(canvas)
for j,index in enumerate(np.linspace(0,len(frames)-1,12,dtype=int)):
    f=frames[index];image=Image.open(__import__("io").BytesIO(z.read(f["file"]))).convert("RGB");image.thumbnail((420,340))
    x=(j%3)*426;y=(j//3)*280;image.thumbnail((420,245));canvas.paste(image,(x,y+25));draw.text((x+5,y+5),f"{(int(f['timestampNs'])-manifest['startedBootNs'])/1e9:.2f} seconds; frame {index}",fill="black")
canvas.save(REVIEW/"contact-sheet.jpg")
detector=cv2.aruco.CharucoDetector(board());corners_per_frame=[];detected=[]
for f in frames:
    image=cv2.imdecode(np.frombuffer(z.read(f["file"]),np.uint8),cv2.IMREAD_GRAYSCALE)
    corners,ids,markers,marker_ids=detector.detectBoard(image)
    corners_per_frame.append(0 if ids is None else len(ids))
    detected.append({"file":f["file"],"corners":corners_per_frame[-1],"markers":0 if marker_ids is None else len(marker_ids)})
summary["boardDetection"]={"framesWithAnyCorners":int((np.array(corners_per_frame)>0).sum()),"framesWithAtLeast16Corners":int((np.array(corners_per_frame)>=16).sum()),"medianCorners":float(np.median(corners_per_frame)),"maxCorners":int(max(corners_per_frame))}
z.close()
(REVIEW/"capture-quality.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
(REVIEW/"board-detections.json").write_text(json.dumps(detected,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))
print("Local-only visual evidence:",REVIEW/"contact-sheet.jpg")


{
  "sourceSha256": "18ca43aa7ab2bc876399958668f16f96dafc2db006f1b7f11a3bcc1ea4bfc4ec",
  "captureId": "calibration-1791297358282",
  "status": "completed",
  "device": {
    "manufacturer": "Xiaomi",
    "model": "M2101K6G",
    "os": "11"
  },
  "nativeDurationSeconds": 40.211841391,
  "cameraDimensions": [
    640,
    480
  ],
  "timestampSource": "realtime",
  "streams": {
    "accelerometer": {
      "readings": 8416,
      "observedHz": 210.40044515129517,
      "maxGapMs": 4.938125,
      "p99GapMs": 4.84582586,
      "nonmonotonic": 0,
      "finite": true,
      "units": "m/s2"
    },
    "gyroscope": {
      "readings": 8415,
      "observedHz": 210.40043430962675,
      "maxGapMs": 4.938125,
      "p99GapMs": 4.84582637,
      "nonmonotonic": 0,
      "finite": true,
      "units": "rad/s"
    },
    "magnetometer": {
      "readings": 2000,
      "observedHz": 50.000000353926964,
      "maxGapMs": 20.0,
      "p99GapMs": 20.0,
      "nonmonotonic": 0,
      "finite": true,

In [13]:
BUNDLE = Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791297358282.zip")
report = process_bundle(BUNDLE,OUTPUT/BUNDLE.stem)
print(json.dumps(report,indent=2))


{
  "schemaVersion": 1,
  "kind": "phone-camera-imu-calibration-v1",
  "bundleSha256": "18ca43aa7ab2bc876399958668f16f96dafc2db006f1b7f11a3bcc1ea4bfc4ec",
  "status": "rejected",
  "distanceReady": false,
  "independentDistanceValidation": false,
  "board": {
    "squaresX": 9,
    "squaresY": 7,
    "squareLengthM": 0.02,
    "markerLengthM": 0.015,
    "dictionary": "DICT_4X4_100"
  },
  "errors": [
    "Held-out gyro/clock alignment did not pass"
  ],
  "binding": {
    "installationId": "3899f646-3d2e-4827-a88a-cc523afe77ab",
    "captureId": "calibration-1791297358282",
    "cameraId": "1",
    "width": 640,
    "height": 480,
    "imageAxes": "native-sensor-unrotated-unmirrored",
    "sensorOrientationDegrees": 270,
    "stabilizationRequested": "off"
  },
  "lens": {
    "cameraMatrix": [
      [
        493.19151074520386,
        0.0,
        316.9271660352964
      ],
      [
        0.0,
        493.3450352936252,
        242.10519260037455
      ],
      [
        0.0,
    

## Interpretation of this actual capture

- Native completed capture: 40.212 s, 499 stored frames, all 499 matching camera-result timestamps; hardware clock source realtime. Stored images ~12.66 Hz, maximum gap 133.2 ms. Accelerometer/gyro ~210.40 Hz with maximum native gap 4.94 ms; magnetometer 50 Hz, 20 ms maximum gap. These are the calibration module's delivered rates, not a claim about the separate gait recorder.
- Board detections: 397/499 frames with corners; 277 with at least 16 corners; maximum 48. The digital target is visible and supports lens fitting. Training reprojection RMS 0.1673 pixels, held-out point-coordinate RMS 90th percentile 0.1547 pixels after correcting the checker below. Intrinsics are a provisional fit for this 640x480 native front-camera pipeline.
- A bug in the original offline reprojection checker broadcast projected (N,1,2) against observed (N,2), creating an N-by-N array and falsely rejecting lens fit. Corrected to compare corresponding flattened Nx2 points; an explicit shape-alias regression now passes. The initial lens rejection is invalid, not evidence of bad capture.
- Final full-profile outcome: rejected at camera/gyro alignment. Held-out angular-rate coordinate RMS 0.09780 rad/s exceeds the current engineering gate 0.08. Estimated residual offset -19.38 ms is a failed-fit diagnostic, not accepted timing calibration. No camera-to-IMU lever arm/accelerometer fit was admitted; distanceReady remains false.
- User confirmed the displayed tablet was held and moved during capture. A fixed board is required for IMU-vs-visual motion fitting. Lens fitting can use relative board poses, but moving the board contaminates the IMU correspondence. User confirmed checking displayed square size at 20 mm; this is operator-reported verification, not an independent measurement by the processor. The target-motion prerequisite is therefore unmet even if a numerical residual gate were to pass.
- First-five-second per-axis gyro RMS 0.3649 rad/s indicates motion during the proposed quiet period. Not a diagnosis of user behavior or cause of alignment failure.

Preserve and reuse this capture for acquisition/lens work. Do not import it as a completed full calibration or use the board-derived distance as validated walking distance. Room images/contact sheet and raw captures remain local and are not embedded in the notebook.


## Second supplied capture: calibration-1791298033545

Executed with the same corrected processing gates; previous capture outputs are retained.


In [15]:
BUNDLE = Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791298033545.zip")
import hashlib, zipfile, numpy as np, cv2
from PIL import Image, ImageDraw
from camera_calibration import read_bundle, board
REVIEW = OUTPUT/BUNDLE.stem
REVIEW.mkdir(parents=True,exist_ok=True)
z, manifest, frames, imu, capture_results = read_bundle(BUNDLE)
summary = {"sourceSha256":hashlib.sha256(BUNDLE.read_bytes()).hexdigest(), "captureId":manifest["captureId"], "status":manifest["status"], "device":manifest["device"], "nativeDurationSeconds":(manifest["endedBootNs"]-manifest["startedBootNs"])/1e9, "cameraDimensions":[manifest["width"],manifest["height"]], "timestampSource":manifest["timestampSource"], "streams":{}}
for sensor in ["accelerometer","gyroscope","magnetometer"]:
    rows=[v for v in imu if v["sensor"]==sensor]
    times=np.array([int(v["timestampNs"]) for v in rows],dtype=np.int64)
    gaps=np.diff(times)/1e6
    xyz=np.array([[v[k] for k in ["x","y","z"]] for v in rows])
    summary["streams"][sensor]={"readings":len(rows), "observedHz":(len(rows)-1)*1e9/(times[-1]-times[0]), "maxGapMs":float(gaps.max()), "p99GapMs":float(np.percentile(gaps,99)), "nonmonotonic":int((gaps<=0).sum()), "finite":bool(np.isfinite(xyz).all()), "units":manifest["units"][sensor]}
    if sensor=="gyroscope":
        start=manifest["startedBootNs"]
        summary["quietFirstFiveSecondsGyroRmsRadS"]=float(np.sqrt(np.mean(xyz[(times-start)<5e9]**2)))
        summary["gyroAxisStdRadS"]=np.std(xyz,axis=0).tolist()
times=np.array([int(v["timestampNs"]) for v in frames],dtype=np.int64)
summary["camera"]={"frames":len(frames), "savedFrameHz":(len(frames)-1)*1e9/(times[-1]-times[0]), "maxGapMs":float((np.diff(times)/1e6).max()), "resultMatches":sum(int(v["timestampNs"]) in capture_results for v in frames)}
canvas=Image.new("RGB",(1280,1120),"white");draw=ImageDraw.Draw(canvas)
for j,index in enumerate(np.linspace(0,len(frames)-1,12,dtype=int)):
    f=frames[index];image=Image.open(__import__("io").BytesIO(z.read(f["file"]))).convert("RGB");image.thumbnail((420,340))
    x=(j%3)*426;y=(j//3)*280;image.thumbnail((420,245));canvas.paste(image,(x,y+25));draw.text((x+5,y+5),f"{(int(f['timestampNs'])-manifest['startedBootNs'])/1e9:.2f} seconds; frame {index}",fill="black")
canvas.save(REVIEW/"contact-sheet.jpg")
detector=cv2.aruco.CharucoDetector(board());corners_per_frame=[];detected=[]
for f in frames:
    image=cv2.imdecode(np.frombuffer(z.read(f["file"]),np.uint8),cv2.IMREAD_GRAYSCALE)
    corners,ids,markers,marker_ids=detector.detectBoard(image)
    corners_per_frame.append(0 if ids is None else len(ids))
    detected.append({"file":f["file"],"corners":corners_per_frame[-1],"markers":0 if marker_ids is None else len(marker_ids)})
summary["boardDetection"]={"framesWithAnyCorners":int((np.array(corners_per_frame)>0).sum()),"framesWithAtLeast16Corners":int((np.array(corners_per_frame)>=16).sum()),"medianCorners":float(np.median(corners_per_frame)),"maxCorners":int(max(corners_per_frame))}
z.close()
(REVIEW/"capture-quality.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
(REVIEW/"board-detections.json").write_text(json.dumps(detected,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))
print("Local-only visual evidence:",REVIEW/"contact-sheet.jpg")


{
  "sourceSha256": "df8a30cbc16ff40d21e8b3096e3f370ac87094b3ad375679998d56cbe8db6314",
  "captureId": "calibration-1791298033545",
  "status": "completed",
  "device": {
    "manufacturer": "Xiaomi",
    "model": "M2101K6G",
    "os": "11"
  },
  "nativeDurationSeconds": 40.241429307,
  "cameraDimensions": [
    640,
    480
  ],
  "timestampSource": "realtime",
  "streams": {
    "accelerometer": {
      "readings": 8417,
      "observedHz": 210.33757000557767,
      "maxGapMs": 4.892813,
      "p99GapMs": 4.8503146,
      "nonmonotonic": 0,
      "finite": true,
      "units": "m/s2"
    },
    "gyroscope": {
      "readings": 8417,
      "observedHz": 210.33757000557767,
      "maxGapMs": 4.892813,
      "p99GapMs": 4.8503146,
      "nonmonotonic": 0,
      "finite": true,
      "units": "rad/s"
    },
    "magnetometer": {
      "readings": 2001,
      "observedHz": 50.0,
      "maxGapMs": 20.0,
      "p99GapMs": 20.0,
      "nonmonotonic": 0,
      "finite": true,
      "units": 

In [16]:
report = process_bundle(BUNDLE,OUTPUT/BUNDLE.stem)
print(json.dumps(report,indent=2))


{
  "schemaVersion": 1,
  "kind": "phone-camera-imu-calibration-v1",
  "bundleSha256": "df8a30cbc16ff40d21e8b3096e3f370ac87094b3ad375679998d56cbe8db6314",
  "status": "rejected",
  "distanceReady": false,
  "independentDistanceValidation": false,
  "board": {
    "squaresX": 9,
    "squaresY": 7,
    "squareLengthM": 0.02,
    "markerLengthM": 0.015,
    "dictionary": "DICT_4X4_100"
  },
  "errors": [
    "Held-out gyro/clock alignment did not pass"
  ],
  "binding": {
    "installationId": "3899f646-3d2e-4827-a88a-cc523afe77ab",
    "captureId": "calibration-1791298033545",
    "cameraId": "1",
    "width": 640,
    "height": 480,
    "imageAxes": "native-sensor-unrotated-unmirrored",
    "sensorOrientationDegrees": 270,
    "stabilizationRequested": "off"
  },
  "lens": {
    "cameraMatrix": [
      [
        497.8586159065087,
        0.0,
        321.00142885674063
      ],
      [
        0.0,
        497.78060299896896,
        244.0644842153622
      ],
      [
        0.0,
    

In [17]:
# Diagnostic motion/timing checks; do not override the calibration gate.
z, m, fs, imu_rows, results = read_bundle(BUNDLE)
start = int(m["startedBootNs"])
per_second=[]
gyro=[v for v in imu_rows if v["sensor"]=="gyroscope"]
for second in range(40):
    rows=[v for v in gyro if second <= (int(v["timestampNs"])-start)/1e9 < second+1]
    xyz=np.array([[v[k] for k in ["x","y","z"]] for v in rows])
    per_second.append({"second":second,"readings":len(rows),"gyroCoordinateRmsRadS":float(np.sqrt(np.mean(xyz**2)))})
values=[]
for f in fs:
    r=results[int(f["timestampNs"])]
    values.append({"second":(int(f["timestampNs"])-start)/1e9,"exposureMs":r["exposureNs"]/1e6,"readoutMs":r["rollingShutterSkewNs"]/1e6,"arrivalLagMs":(int(f["arrivalBootNs"])-int(f["timestampNs"]))/1e6})
diagnostics={"firstFiveSeconds":per_second[:5],"exposureMsRange":[min(v["exposureMs"] for v in values),max(v["exposureMs"] for v in values)],"readoutMsRange":[min(v["readoutMs"] for v in values),max(v["readoutMs"] for v in values)],"medianFrameArrivalLagMs":float(np.median([v["arrivalLagMs"] for v in values])),"fullCalibrationStatus":report["status"],"angularFitGateExceededRadS":report["cameraImuRotationTiming"]["heldOutGyroRmseRadS"]-.08,"interpretation":"Near-threshold fit; rolling-shutter/exposure and visual-pose approximation warrant investigation. Cause not established; raw hardware clocks retained."}
(REVIEW/"timing-motion-diagnostics.json").write_text(json.dumps(diagnostics,indent=2),encoding="utf-8")
z.close()
print(json.dumps(diagnostics,indent=2))


{
  "firstFiveSeconds": [
    {
      "second": 0,
      "readings": 210,
      "gyroCoordinateRmsRadS": 0.43049346731858934
    },
    {
      "second": 1,
      "readings": 210,
      "gyroCoordinateRmsRadS": 0.19367524991677315
    },
    {
      "second": 2,
      "readings": 210,
      "gyroCoordinateRmsRadS": 0.0994858722574506
    },
    {
      "second": 3,
      "readings": 211,
      "gyroCoordinateRmsRadS": 0.03002125829006272
    },
    {
      "second": 4,
      "readings": 211,
      "gyroCoordinateRmsRadS": 0.016142985233242726
    }
  ],
  "exposureMsRange": [
    30.0,
    33.333332
  ],
  "readoutMsRange": [
    32.203217,
    32.203217
  ],
  "medianFrameArrivalLagMs": 75.008309,
  "fullCalibrationStatus": "rejected",
  "angularFitGateExceededRadS": 0.00037809639911763615,
  "interpretation": "Near-threshold fit; rolling-shutter/exposure and visual-pose approximation warrant investigation. Cause not established; raw hardware clocks retained."
}



## Second calibration capture - calibration-1791298033545

Executed the same corrected gates on the second user ZIP (SHA-256 df8a30cbc16ff40d21e8b3096e3f370ac87094b3ad375679998d56cbe8db6314), preserving first-capture outputs. Completed 40.241 seconds; 500 640x480 front-camera images, all with matching native capture results, saved-frame rate 12.664 Hz, maximum image gap 233.1 ms. Accelerometer/gyroscope each 8417 samples at 210.338 Hz, maximum gaps 4.893 ms; magnetometer 2001 samples, 50 Hz, 20 ms maximum gap. Finite, strictly monotonic native data. No save/copy failure.

Board corners detected in 333 frames, 226 with at least 16. Provisional lens training RMS 0.12265 pixels, held-out coordinate RMS p90 0.11681 pixels (previous 0.15472). Camera/gyro held-out coordinate RMS improves from 0.09780 to 0.080378 rad/s; existing gate remains 0.0800, so full profile stays rejected, distanceReady=false. The exceedance is only 0.000378 rad/s (0.47%); this is a borderline engineering fit, not evidence of clinically meaningful sensor failure. Threshold is not relaxed to admit the sample. Clock offset estimate -31.02 ms remains a failed-fit diagnostic, not accepted calibration.

Timing diagnostics: exposure 30-33.33 ms, rolling readout 32.203 ms; capture uses an approximate frame midpoint model. Native clocks are retained; median image callback arrival lag 75.0 ms must not be substituted for hardware time. These approximations/visual pose noise are plausible contributors, not an established cause. Early motion settles from gyro coordinate RMS 0.4305 in second 0 to 0.01614 in second 4; the nominal initial five seconds are not entirely still.

Visual contact sheet shows the tablet resting against furniture/support rather than visibly being held; target stationarity cannot be independently guaranteed from sampled images. Earlier operator-verified 20 mm display scale is contextual, not remeasured by this processor. Keep this useful capture for processing/intrinsics; no repeat requested solely for this near-threshold residual. Full rotation/timing/lever-arm validation and independent walking distance remain incomplete. Raw ZIP/images/generated reports remain local, notebook contains executed outputs and interpretations. No source push/OTA.


In [18]:
BUNDLE = Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791298033545.zip")
report = process_bundle(BUNDLE,OUTPUT/BUNDLE.stem/"row-aware-investigation")
print(json.dumps(report,indent=2))


{
  "schemaVersion": 1,
  "kind": "phone-camera-imu-calibration-v1",
  "processorVersion": "2-row-aware",
  "bundleSha256": "df8a30cbc16ff40d21e8b3096e3f370ac87094b3ad375679998d56cbe8db6314",
  "status": "rejected",
  "distanceReady": false,
  "independentDistanceValidation": false,
  "board": {
    "squaresX": 9,
    "squaresY": 7,
    "squareLengthM": 0.02,
    "markerLengthM": 0.015,
    "dictionary": "DICT_4X4_100"
  },
  "errors": [
    "Held-out gyro/clock alignment did not pass"
  ],
  "binding": {
    "installationId": "3899f646-3d2e-4827-a88a-cc523afe77ab",
    "captureId": "calibration-1791298033545",
    "cameraId": "1",
    "width": 640,
    "height": 480,
    "imageAxes": "native-sensor-unrotated-unmirrored",
    "sensorOrientationDegrees": 270,
    "stabilizationRequested": "off"
  },
  "lens": {
    "cameraMatrix": [
      [
        497.8586159468034,
        0.0,
        321.0014288726686
      ],
      [
        0.0,
        497.78060303927447,
        244.064484229201

In [20]:
from camera_imu_joint import load_board_views, fit_joint_rotation_timing
BUNDLE=Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791298033545.zip")
BASE=OUTPUT/BUNDLE.stem
previous=json.loads((BASE/"calibration-report.json").read_text())
views,gt,gv,K,D=load_board_views(BUNDLE,previous["lens"])
row_joint=fit_joint_rotation_timing(views,gt,gv,K,D,previous["cameraImuRotationTiming"])
(BASE/"joint-row-timing-report.json").write_text(json.dumps(row_joint,indent=2),encoding="utf-8")
print(json.dumps(row_joint,indent=2))


{
  "method": "row-timed-gyro-pixel-fit-v1",
  "processorSha256": "1a1934bdb1b1d18d1a6895712311305dd3dd10d26b8c58129d947fa68dafdea3",
  "status": "rotation-timing-candidate",
  "fullCalibrationReady": false,
  "rotationImuToCamera": [
    [
      0.4461503023550424,
      0.8916250816308098,
      0.07716619412253135
    ],
    [
      -0.8935948982814359,
      0.44856887055227934,
      -0.016556755021525513
    ],
    [
      -0.04937677058997082,
      -0.061568516128816975,
      0.9968807613490196
    ]
  ],
  "rotationWorldImuAtGyroOrigin": [
    [
      -0.9813688585209779,
      0.09844183690140437,
      0.1649980856637745
    ],
    [
      -0.18791336350743135,
      -0.6707677877833436,
      -0.7174671718535723
    ],
    [
      0.04004661459545539,
      -0.7351052847175816,
      0.6767691549115742
    ]
  ],
  "gyroBiasRadS": [
    -0.005600625096237901,
    -0.0007241645415657476,
    -0.0029417374889698688
  ],
  "residualTimeOffsetSeconds": -0.029728361132368618,
 

In [21]:
from camera_calibration import read_bundle
from camera_imu_joint import screen_board_acceleration
import numpy as np
z,m,frames,imu,results=read_bundle(BUNDLE)
z.close()
origin=int(m["startedBootNs"])
accel=[v for v in imu if v["sensor"]=="accelerometer"]
at=np.array([(int(v["timestampNs"])-origin)/1e9 for v in accel])
av=np.array([[v[k] for k in ["x","y","z"]] for v in accel])
accel_screen=screen_board_acceleration(views,gt,gv,K,D,row_joint,at,av)
(BASE/"joint-acceleration-screen.json").write_text(json.dumps(accel_screen,indent=2),encoding="utf-8")
print(json.dumps(accel_screen,indent=2))


{
  "passed": false,
  "reason": "No sufficiently long contiguous board tracking segments; no interpolation across losses",
  "acceptedPoseViews": 218,
  "maximumAcceptedPoseGapSeconds": 2.1360120773315447
}


In [22]:
from camera_imu_joint import load_board_views, fit_joint_rotation_timing
first_bundle=Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791297358282.zip")
first_base=OUTPUT/first_bundle.stem
first_previous=json.loads((first_base/"calibration-report.json").read_text())
v1,t1,g1,K1,D1=load_board_views(first_bundle,first_previous["lens"])
negative=fit_joint_rotation_timing(v1,t1,g1,K1,D1,first_previous["cameraImuRotationTiming"])
negative["knownMovingTarget"]=True
(first_base/"joint-moving-target-negative-control.json").write_text(json.dumps(negative,indent=2),encoding="utf-8")
print(json.dumps(negative,indent=2))
assert not negative["passed"]


{
  "method": "row-timed-gyro-pixel-fit-v1",
  "processorSha256": "1a1934bdb1b1d18d1a6895712311305dd3dd10d26b8c58129d947fa68dafdea3",
  "status": "rejected",
  "fullCalibrationReady": false,
  "rotationImuToCamera": [
    [
      -0.4516286753416953,
      0.8552963041084928,
      0.2539680526906055
    ],
    [
      -0.8394486421196837,
      -0.5037765484916548,
      0.2038017822131462
    ],
    [
      0.3022540601092478,
      -0.12115040803970917,
      0.9454972563573556
    ]
  ],
  "rotationWorldImuAtGyroOrigin": [
    [
      -0.8072374571472007,
      -0.5895114135202177,
      0.029051352944681502
    ],
    [
      0.4841459461878532,
      -0.6895039021793843,
      -0.5386901443958955
    ],
    [
      0.33759500969118295,
      -0.42078576759296094,
      0.8420029377756428
    ]
  ],
  "gyroBiasRadS": [
    0.005560935716609936,
    0.011956536529086006,
    0.002151936027565409
  ],
  "residualTimeOffsetSeconds": -0.0576372291055247,
  "trainingCoordinateRmsPixels

In [24]:
import cv2, numpy as np
from camera_calibration import read_bundle,camera_observation_time_seconds
from visual_tracking import track_native_frames
BUNDLE=Path(r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791298033545.zip")
BASE=OUTPUT/BUNDLE.stem
lens=json.loads((BASE/"calibration-report.json").read_text())["lens"]
z,m,frames,imu,results=read_bundle(BUNDLE)
def native_images():
    for f in frames:
        t=camera_observation_time_seconds(f,results[int(f["timestampNs"])],int(m["startedBootNs"]),0,m["height"])
        yield t,cv2.imdecode(np.frombuffer(z.read(f["file"]),np.uint8),0)
try:
    tracking=track_native_frames(native_images(),np.asarray(lens["cameraMatrix"]),np.asarray(lens["distortion"]))
    tracking_screen={"status":"visual-trajectory-candidate","frames":len(tracking["times"]),"distanceReady":False,"limitations":tracking["limitations"]}
except ValueError as e:
    tracking_screen={"status":"rejected","reason":str(e),"distanceReady":False}
finally:
    z.close()
(BASE/"visual-tracking-screen.json").write_text(json.dumps(tracking_screen,indent=2),encoding="utf-8")
print(json.dumps(tracking_screen,indent=2))


{
  "status": "rejected",
  "reason": "Tracking lost: inconsistent map pose",
  "distanceReady": false
}


## Current result and boundary

Actual row-timed joint fitting on the second capture passes internal rotation/timing gates: held-out coordinate RMS **1.1594 px**, p90 **1.6068 px**, candidate offset **-29.73 ms**. It does not pass a full calibration or distance validation. The confirmed moving-tablet first capture fails the same model (**6.9382 px** RMS); its optimizer also reaches the evaluation limit. Mean-row correction alone failed the original angular-rate gate (**0.08162 rad/s**), which was not relaxed.

The translation screen finds 218 usable poses but no sufficiently long uninterrupted segment; maximum accepted-pose gap is **2.136 s**. No acceleration/lever-arm result is fabricated. The conservative visual frontend loses its fixed map on the second bench capture. This actual failure is retained; neither capture is a walking accuracy test. Second-capture target stationarity is not operator-confirmed.

Reusable visual tracking and metric-scale initialization now exist locally. They use consistent monocular map poses and SI IMU samples, not an entered route length or a step-length formula. Synthetic tests recover known scale and reject unobservable, inconsistent, pure-rotation and missing-data cases. These tests verify implementation under their stated models, not device accuracy. The route connector is not a continuous VIO backend, lacks rolling-shutter visual correction/map expansion/loop closure and is not installed in Android. Any camera path estimate may contain residual body sway.

No new capture is requested merely to fix the old borderline gyro gate. Do not import the rotation-only report, enable distance stopping, or claim full calibration. App version/runtime and clinical test behavior are unchanged.


## Reuse and validation boundary

Import calibration-report.json using Import processed calibration on the same phone. Accepted reports remain geometry-candidate, distanceReady=false. They bind the installation, camera ID, image dimensions/axes, orientation and stabilization setting. They cannot be transferred to another phone or applied directly to the older Expo MP4 pipeline: its crop/rotation, resolution and frame clocks differ. Reinstallation clears the binding. Changing holder optics/focus/capture settings requires rechecking. Patient-specific standing midline calibration remains separate.

A local fixed-map tracking/metric-scale research prototype is provided, but no complete native camera odometry engine or independently validated walking distance. Next gate: implement the same native frame pipeline for walking tracking, then compare unseen measured routes, turns and repeated captures against ground truth before enabling an automatic 10MWT boundary. Do not reuse the same 3 m route as both scale fitting and accuracy evidence.

Method sources: [OpenVINS calibration](https://docs.openvins.com/gs-calibration.html), [Android camera clock contract](https://developer.android.com/reference/android/hardware/camera2/CameraCharacteristics#SENSOR_INFO_TIMESTAMP_SOURCE_REALTIME), [OpenCV camera calibration](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html).

## 7 October: native app research trial verification

The research profile is a separate rotation-only candidate, not accepted full geometry. Import it once on the originating installation using **Camera distance trial > Import phone research profile**. The trial is outside clinical tests. No entered route length supplies metric scale. The following Java tests execute the application algorithm using the matching **real desktop OpenCV JNI** library and a desktop loader adapter. They do not execute Android camera/permission APIs or establish physical walking accuracy.


In [26]:
from distance_research_profile import export_research_profile
BASE=OUTPUT/"calibration-1791298033545"
profile_file=BASE/"GaitTrace-phone-research-profile.json"
research_profile=export_research_profile(json.loads((BASE/"calibration-report.json").read_text()),json.loads((BASE/"joint-row-timing-report.json").read_text()),profile_file)
print(json.dumps({"file":str(profile_file.relative_to(ROOT)),"fullCalibrationReady":research_profile["fullCalibrationReady"],"distanceReady":research_profile["distanceReady"]},indent=2))


{
  "file": "android\\dist\\camera-calibration\\calibration-1791298033545\\GaitTrace-phone-research-profile.json",
  "fullCalibrationReady": false,
  "distanceReady": false
}


In [27]:
import os, subprocess, hashlib
from native_test_fixture import write_native_fixture
TOOLS=ROOT/"android/dist/native-distance-tools"
JDK=next(v for v in sorted(TOOLS.glob("jdk*")) if v.is_dir() and (v/"bin/javac.exe").exists())
fixture_file=write_native_fixture(TOOLS/"metric-fixture.json")
classpath=os.pathsep.join(str(TOOLS/name) for name in ["opencv.jar","commons-math3.jar","json.jar"])
classes=TOOLS/"test-classes";classes.mkdir(exist_ok=True)
java_source=ROOT/"android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/ResearchDistanceTracker.java"
test_source=ROOT/"android/research/native-tests"
subprocess.run([str(JDK/"bin/javac.exe"),"-cp",classpath,"-d",str(classes),str(test_source/"OpenCVLoader.java"),str(java_source),str(test_source/"NativeDistanceAlgorithmCheck.java"),str(test_source/"NativeDistanceCaptureReplay.java")],check=True,capture_output=True,text=True)
java_command=[str(JDK/"bin/java.exe"),"-Djava.library.path="+str(TOOLS/"windows-opencv"),"-cp",str(classes)+os.pathsep+classpath]
native_dll=TOOLS/"windows-opencv/opencv_java4120.dll"
check=subprocess.run(java_command+["NativeDistanceAlgorithmCheck",str(native_dll),str(fixture_file)],check=True,capture_output=True,text=True)
native_checks=json.loads(check.stdout);native_checks["sourceSha256"]=hashlib.sha256(java_source.read_bytes()).hexdigest()
(TOOLS/"native-known-answer-checks.json").write_text(json.dumps(native_checks,indent=2))
print(json.dumps(native_checks,indent=2))


{
  "androidCaptureExecuted": false,
  "knownAnswerChecks": 5,
  "physicalDeviceValidation": false,
  "heldOutRms": 0.002198795229105009,
  "nativeMetricScale": 2.7300776846203405,
  "jniVersion": "4.12.0",
  "expectedScale": 2.7,
  "sourceSha256": "7e53fdd42537dd0b0d00c83f9cd4507d2d2a09c2f7d441cb466cee338ba0076f"
}


In [28]:
# Explicit artifact replay: all IMU data is available before images; live latency is untested.
replay_file=TOOLS/"native-artifact-replay.json"
replay=subprocess.run(java_command+["NativeDistanceCaptureReplay",str(native_dll),r"C:/Users/frank/Downloads/Telegram Desktop/calibration-1791298033545.zip",str(profile_file),str(replay_file)],check=True,capture_output=True,text=True)
print(replay.stdout)


{
  "androidCaptureExecuted": false,
  "reason": "inconsistent-map-pose",
  "features": 114,
  "artifactReplay": true,
  "poseFrames": 1,
  "reprojectionRmsPixels": 0,
  "framesProcessed": 443,
  "status": "rejected"
}



## Physical app check still required

Desktop JNI checks passed, but actual second-capture replay lost its fixed map after one pose (443 frames processed after the nominal five-second bench phase). This is not a measured walking-route validation. The Android trial preserves raw frames, all three sensor streams, frame metadata, research profile, release identifiers and tracking diagnostics in one ZIP. Import the matching profile once, start a marked short route, place the phone, wait for the voice cue, walk and stand still until the finish voice. Export the trial ZIP. No manual reach-to-screen stop is required in normal flow.

Any estimate is experimental horizontal camera travel over the tracked interior segment: initialization/derivative edges and body sway mean it cannot yet represent validated whole-route distance or clinical gait speed. Do not enable automatic clinical distance boundaries. Physical rates/gaps, timing, setup/movement/quiet phases, speech, holder image quality and paired reference-distance errors remain to be checked on the installed APK.


## First actual 3 m walking trial ? 7 October 2026

User reports a 3 m walk. The route length has not been independently measured here. This is the first uploaded physical build-10 distance-trial capture, not a simulated capture. ZIP integrity and source hash are checked. The device had **no imported research profile**, so its tracker remained `needs-profile`, with no poses or metric distance. Raw images and all three sensors remain available.


In [29]:
# New execution on the first uploaded measured-route artifact, not a live capture.
from pathlib import Path
import json, zipfile, hashlib, os, subprocess
import numpy as np
import cv2
from PIL import Image, ImageDraw
ROOT = Path.cwd() if (Path.cwd()/"android").exists() else Path.cwd().parent
trial_zip = Path(r"C:/Users/frank/Downloads/Telegram Desktop/distance-trial-1791347211905.zip")
trial_out = ROOT/"android/dist/camera-distance/distance-trial-1791347211905"
trial_out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(trial_zip) as archive:
    assert archive.testzip() is None
    manifest = json.loads(archive.read("manifest.json"))
    diagnostic = json.loads(archive.read("distance-diagnostics.json"))
    frames = [json.loads(v) for v in archive.read("frames.ndjson").decode().splitlines()]
    imu = [json.loads(v) for v in archive.read("imu.ndjson").decode().splitlines()]
    camera_results = [json.loads(v) for v in archive.read("capture-results.ndjson").decode().splitlines()]
    walk = diagnostic["walkStartedBootNs"]
    report = {"sourceSha256": hashlib.file_digest(trial_zip.open("rb"), "sha256").hexdigest(),
        "referenceRouteMetresOperatorReported": 3, "independentRouteMeasurementVerified": False,
        "captureSeconds": (manifest["endedBootNs"]-manifest["startedBootNs"])/1e9,
        "movementTriggerSecondsAfterCapture": (walk-manifest["startedBootNs"])/1e9,
        "movementTriggerToFinishSeconds": (manifest["endedBootNs"]-walk)/1e9,
        "profileIncluded": "research-profile.json" in archive.namelist(),
        "appRelease": manifest["appRelease"], "onDeviceStatus": diagnostic["status"],
        "onDeviceReason": diagnostic["reason"], "sensorStats": {}}
    for sensor in ("accelerometer", "gyroscope", "magnetometer"):
        rows = [v for v in imu if v["sensor"] == sensor]
        t = np.array([v["timestampNs"] for v in rows], dtype=np.int64)
        values = np.array([[v[a] for a in "xyz"] for v in rows])
        gaps = np.diff(t)/1e6
        report["sensorStats"][sensor] = {"count": len(rows),
            "meanHz": (len(t)-1)/((t[-1]-t[0])/1e9), "maxGapMs": float(gaps.max()),
            "p99GapMs": float(np.percentile(gaps,99)), "strictlyMonotonic": bool(np.all(gaps>0)),
            "allFinite": bool(np.isfinite(values).all()), "walkingPhaseCount": int(np.sum(t>=walk))}
    t = np.array([v["timestampNs"] for v in frames], dtype=np.int64)
    camera = {v["timestampNs"]: v for v in camera_results}
    matched = [camera[v["timestampNs"]] for v in frames if v["timestampNs"] in camera]
    report["camera"] = {"frames": len(frames), "meanStoredHz": (len(t)-1)/((t[-1]-t[0])/1e9),
        "maxGapMs": float(np.diff(t).max()/1e6), "matchedCaptureResults": len(matched),
        "medianArrivalDelayMs": float(np.median([(v["arrivalBootNs"]-v["timestampNs"])/1e6 for v in frames])),
        "exposureRangeMs": [min(v["exposureNs"] for v in matched)/1e6, max(v["exposureNs"] for v in matched)/1e6],
        "readoutRangeMs": [min(v["rollingShutterSkewNs"] for v in matched)/1e6, max(v["rollingShutterSkewNs"] for v in matched)/1e6]}
    motion = []
    for second in range(-2, int(np.ceil(report["movementTriggerToFinishSeconds"]))):
        row = {"secondsFromMovementTrigger": second}
        for sensor in ("accelerometer", "gyroscope"):
            values = np.array([[v[a] for a in "xyz"] for v in imu if v["sensor"]==sensor and second<=(v["timestampNs"]-walk)/1e9<second+1])
            if len(values):
                norm = np.linalg.norm(values,axis=1)
                row[sensor+"NormStd"] = float(np.std(norm))
                row[sensor+"NormMean"] = float(np.mean(norm))
        motion.append(row)
    report["oneSecondMotionWindows"] = motion
    sheet = Image.new("RGB",(960,810),(245,245,245)); draw = ImageDraw.Draw(sheet)
    for j,relative in enumerate([-1,0,.8,1.6,2.4,3.2,5,8,11]):
        frame = min(frames,key=lambda v:abs((v["timestampNs"]-walk)/1e9-relative))
        bgr = cv2.imdecode(np.frombuffer(archive.read(frame["file"]),np.uint8),cv2.IMREAD_COLOR)
        image = Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)); image.thumbnail((320,240))
        x=(j%3)*320; y=(j//3)*270; sheet.paste(image,(x,y+25))
        draw.text((x+6,y+7),f'{(frame["timestampNs"]-walk)/1e9:.2f}s / {frame["file"]}',fill="black")
    sheet.save(trial_out/"walk-contact-sheet.jpg")
(trial_out/"capture-audit.json").write_text(json.dumps(report,indent=2))
print(json.dumps({k:v for k,v in report.items() if k!="oneSecondMotionWindows"},indent=2))


{
  "sourceSha256": "0636e70d9e720042081a1dac6bae75adf585c99fab21945e5a385d1a83d6b588",
  "referenceRouteMetresOperatorReported": 3,
  "independentRouteMeasurementVerified": false,
  "captureSeconds": 21.594261398,
  "movementTriggerSecondsAfterCapture": 9.227388402,
  "movementTriggerToFinishSeconds": 12.366872996,
  "profileIncluded": false,
  "appRelease": {
    "appVersion": "0.3.3",
    "buildNumber": "10",
    "runtimeVersion": "037f976718a2d2ef78c79b8705adeb04596f072c",
    "updateId": "759e4e48-f9f5-47eb-897c-f2b86a8d3756",
    "channel": "preview",
    "embedded": true
  },
  "onDeviceStatus": "unavailable",
  "onDeviceReason": "needs-profile",
  "sensorStats": {
    "accelerometer": {
      "count": 4510,
      "meanHz": 210.40092452056996,
      "maxGapMs": 4.895938,
      "p99GapMs": 4.844739919999999,
      "strictlyMonotonic": true,
      "allFinite": true,
      "walkingPhaseCount": 2566
    },
    "gyroscope": {
      "count": 4510,
      "meanHz": 210.40092452056996,
 

In [30]:
# Supply the existing matching research profile OFFLINE; it was absent in the device trial.
# All raw IMU samples are loaded before images. This does not validate live tracking latency.
TOOLS = ROOT/"android/dist/native-distance-tools"
JDK = next(v for v in sorted(TOOLS.glob("jdk*")) if v.is_dir() and (v/"bin/java.exe").exists())
classpath = os.pathsep.join(str(TOOLS/v) for v in ["test-classes","opencv.jar","commons-math3.jar","json.jar"])
profile_file = ROOT/"android/dist/camera-calibration/calibration-1791298033545/GaitTrace-phone-research-profile.json"
replay_file = trial_out/"profile-artifact-replay.json"
replay = subprocess.run([str(JDK/"bin/java.exe"),"-Djava.library.path="+str(TOOLS/"windows-opencv"),"-cp",classpath,
    "NativeDistanceCaptureReplay",str(TOOLS/"windows-opencv/opencv_java4120.dll"),str(trial_zip),str(profile_file),str(replay_file)],
    check=True,capture_output=True,text=True)
r = json.loads(replay_file.read_text())
pose_times = [v["timeSeconds"] for v in r["trajectoryMapUnits"]]
post_trigger = [v for v in frames if v["timestampNs"]>=walk]
failed_index = next(i for i,v in enumerate(r["frameDiagnostics"]) if v["reason"])
summary = {"status": r["status"], "reason": r["reason"], "poseFrames": r["poseFrames"],
    "replayedFrames": r["replayedFrames"], "trackedPoseSpanSeconds": pose_times[-1]-pose_times[0],
    "failureSecondsAfterMovementTrigger": (post_trigger[failed_index]["timestampNs"]-walk)/1e9,
    "initialFeatures": r["frameDiagnostics"][0]["features"], "lastFeatures": r["features"],
    "lastReprojectionRmsPixels": r["reprojectionRmsPixels"],
    "artifactReplay": True, "allImuAvailableBeforeFrames": True,
    "onDeviceProfileIncluded": False, "validMetricDistanceProduced": False,
    "algorithmSourceSha256": hashlib.sha256((ROOT/"android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/ResearchDistanceTracker.java").read_bytes()).hexdigest()}
(trial_out/"profile-replay-summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


{
  "status": "rejected",
  "reason": "inconsistent-map-pose",
  "poseFrames": 20,
  "replayedFrames": 154,
  "trackedPoseSpanSeconds": 1.4987105729996983,
  "failureSecondsAfterMovementTrigger": 1.839519379,
  "initialFeatures": 413,
  "lastFeatures": 30,
  "lastReprojectionRmsPixels": 1.2474015965141736,
  "artifactReplay": true,
  "allImuAvailableBeforeFrames": true,
  "onDeviceProfileIncluded": false,
  "validMetricDistanceProduced": false,
  "algorithmSourceSha256": "7e53fdd42537dd0b0d00c83f9cd4507d2d2a09c2f7d441cb466cee338ba0076f"
}


## Interpretation and next engineering gap

Capture succeeds: 270 images with matching native result times, approximately 12.82 stored frames/s, 4,510 samples each of acceleration/gyro at 210.40 Hz, and 1,072 magnetic samples at 50 Hz. All samples are finite and strictly monotonic, maximum sensor gaps 4.90/4.90/20.00 ms. The capture lasts 21.59 s, movement trigger at 9.23 s, automatic quiet-stop 12.37 s later. The finish includes stillness and must not be used as walking time or as speed denominator. Substantial motion already exists in the two seconds before the movement trigger; the data does not establish its first-footstep boundary.

Offline replay supplies the existing phone-bound rotation-only profile, without fitting the supplied 3 m reference. It initially finds 413 features, forms 20 map poses over 1.50 s, falls to 30 features and rejects `inconsistent-map-pose` 1.84 s after the recorded trigger. No metric scale/distance is accepted. This improves on the earlier bench replay's one pose, but is still a failure of this fixed-map prototype, not 3 m validation. All IMU samples are available before frames in the replay, so live latency remains untested.

The locally inspected contact sheet shows right-edge/upper holder obstruction, textured room features and motion blur. It does not establish that obstruction alone caused failure. At this initial review, the tracker retained one fixed map and lacked keyframe/map renewal or relocalization. Required improvements before repeating: make missing-profile status an explicit prerequisite for a distance-estimation trial (raw-only acquisition must be clearly distinguished), then improve map continuity and evaluate it against this retained walk. Also investigate pre-trigger motion/start-cue timing rather than claiming exact first-step onset. Neither importing the profile nor relaxing acceptance thresholds is a complete fix. Clinical endpoints and distanceReady remain false. Images, user profile, ZIP and generated audit/replay outputs stay local and ignored.


## Map renewal and profile prerequisite - local 0.3.4 retry

The previous actual walk exposed a missing profile and fixed-map feature depletion. The revised Java tracker detects candidate features at accepted camera poses, tracks them forward/backward, and triangulates new world landmarks using two accepted poses. It retains the existing map scale and coordinate frame, enforces positive depth, parallax and reprojection checks, and never resets/stitches scales after loss. Existing PnP, clock, IMU and physical scale gates remain unchanged. Native and UI entry now reject a missing phone profile before capture begins, with English/Malay/Chinese prompts.


In [31]:
# New execution of revised source on known numerical fixtures and the retained walk.
# The walk is development evidence; its user-reported 3 m is NOT a scale-fit input.
from pathlib import Path
import json,os,subprocess,hashlib
ROOT=Path.cwd() if (Path.cwd()/"android").exists() else Path.cwd().parent
TOOLS=ROOT/"android/dist/native-distance-tools"
JDK=next(v for v in sorted(TOOLS.glob("jdk*")) if v.is_dir() and (v/"bin/javac.exe").exists())
classes=TOOLS/"test-classes"
classpath=os.pathsep.join(str(TOOLS/v) for v in ["opencv.jar","commons-math3.jar","json.jar"])
source=ROOT/"android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/ResearchDistanceTracker.java"
tests=ROOT/"android/research/native-tests"
subprocess.run([str(JDK/"bin/javac.exe"),"-cp",classpath,"-d",str(classes),str(tests/"OpenCVLoader.java"),str(source),str(tests/"NativeDistanceAlgorithmCheck.java"),str(tests/"NativeDistanceCaptureReplay.java")],check=True,capture_output=True,text=True)
command=[str(JDK/"bin/java.exe"),"-Djava.library.path="+str(TOOLS/"windows-opencv"),"-cp",str(classes)+os.pathsep+classpath]
dll=TOOLS/"windows-opencv/opencv_java4120.dll"
checks=subprocess.run(command+["NativeDistanceAlgorithmCheck",str(dll),str(TOOLS/"metric-fixture.json")],check=True,capture_output=True,text=True)
known=json.loads(checks.stdout);known["sourceSha256"]=hashlib.sha256(source.read_bytes()).hexdigest()
(TOOLS/"growing-map-known-answer-checks.json").write_text(json.dumps(known,indent=2))
trial_out=ROOT/"android/dist/camera-distance/distance-trial-1791347211905"
replay_path=trial_out/"growing-map-replay.json"
subprocess.run(command+["NativeDistanceCaptureReplay",str(dll),r"C:/Users/frank/Downloads/Telegram Desktop/distance-trial-1791347211905.zip",str(ROOT/"android/dist/camera-calibration/calibration-1791298033545/GaitTrace-phone-research-profile.json"),str(replay_path)],check=True,capture_output=True,text=True)
r=json.loads(replay_path.read_text())
summary={k:v for k,v in r.items() if k not in ["trajectoryMapUnits","frameDiagnostics","limitations"]}
summary["trackedPoseSpanSeconds"]=r["trajectoryMapUnits"][-1]["timeSeconds"]-r["trajectoryMapUnits"][0]["timeSeconds"]
summary["algorithmSourceSha256"]=known["sourceSha256"]
summary["developmentArtifactReplay"]=True
summary["referenceDistanceUsedForFit"]=False
(trial_out/"growing-map-replay-summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps({"knownAnswerChecks":known,"actualWalkReplay":summary},indent=2))


{
  "knownAnswerChecks": {
    "androidCaptureExecuted": false,
    "knownAnswerChecks": 6,
    "physicalDeviceValidation": false,
    "heldOutRms": 0.002198795229105009,
    "nativeMetricScale": 2.7300776846203405,
    "jniVersion": "4.12.0",
    "expectedScale": 2.7,
    "sourceSha256": "db8082c1de30f17a5687dd204c92f1ec81dda6bfc68b0f31177ddd553ca44b4b"
  },
  "actualWalkReplay": {
    "keyframes": 8,
    "androidCaptureExecuted": false,
    "reason": "scale-physical-check-failed",
    "accelerometerBiasMps2": [
      6.232747737618408,
      -1.1020035754831863,
      -5.7139606804109375
    ],
    "reprojectionRmsPixels": 0.8561645495266719,
    "replayedFrames": 154,
    "heldOutCoordinateRmsMps2": 0.18856784737945287,
    "features": 60,
    "scaledCondition": 104.48211584907324,
    "independentDistanceValidation": false,
    "state": "tracking",
    "artifactReplay": true,
    "gravityWorldMps2": [
      -3.34817608648569,
      2.9626838849951183,
      -7.267142893126305
    ]

The retained development walk now yields 151 poses spanning 11.823 s, with 318 new map points and eight candidate keyframes. It no longer loses tracking on this artifact. Scale still rejects physical consistency: fitted acceleration bias is implausible, fitted gravity magnitude/lever arm and scale uncertainty do not meet existing bounds. The small held-out force residual does not override these physical gates. **No accepted metre estimate or 3 m error is available.** This is development replay with all IMU available before images, not live latency verification or independent clinical accuracy. Six actual JNI numerical checks pass, including renewed landmark coordinate/scale preservation. A new native APK is required to test live map renewal; build 10 cannot receive Java changes by OTA. Do not request a longer walk as a substitute for addressing calibration/scale observability.


## Native retry package ready

Final 0.3.4/build 11 completed successfully on 7 October 2026 (EAS
784326fd-4a29-4477-a61b-58f93000b00f). Downloaded APK:
`android/dist/apk/GaitTrace-0.3.4-build11.apk`, 233,322,494 bytes, SHA-256
93c94fa267bd4ecea6083c374e2d1f69a2829a92725c3401ff51cbc3335081fa.
Compiled package/version, ZIP integrity, revised growing-map method/profile
prerequisite, four OpenCV ABIs and license assets verified. Runtime remains
36ef802b5c8463d2c0a172abd85bb56d75f0ea0e. Install over the existing app,
open Research tools > Camera distance trial, import the same phone research
profile once if absent, then Start and follow the cue. Walk the measured short
route, stand still until finish and export the diagnostic ZIP. Physical retry
and metric accuracy remain pending. No Git push or new OTA has been made.

APK/package verification is build evidence, not a newly validated walking-distance result. Source publication remains pending required exact-file review.


## Second actual phone trial - build 11 with imported profile

User asks whether trial distance-trial-1791349028491 is better. Its route length was not restated, so no reference distance or route error is assumed. Device exports identify 0.3.4/build 11 and growing-map v2. Unlike the previous real trial, the matching research profile is included. Native diagnostics, not desktop replay, determine whether the phone worked. All artifacts remain local and ignored.


In [32]:
# New execution on the first uploaded measured-route artifact, not a live capture.
from pathlib import Path
import json, zipfile, hashlib, os, subprocess
import numpy as np
import cv2
from PIL import Image, ImageDraw
ROOT = Path.cwd() if (Path.cwd()/"android").exists() else Path.cwd().parent
trial_zip = Path(r"C:/Users/frank/Downloads/Telegram Desktop/distance-trial-1791349028491.zip")
trial_out = ROOT/"android/dist/camera-distance/distance-trial-1791349028491"
trial_out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(trial_zip) as archive:
    assert archive.testzip() is None
    manifest = json.loads(archive.read("manifest.json"))
    diagnostic = json.loads(archive.read("distance-diagnostics.json"))
    frames = [json.loads(v) for v in archive.read("frames.ndjson").decode().splitlines()]
    imu = [json.loads(v) for v in archive.read("imu.ndjson").decode().splitlines()]
    camera_results = [json.loads(v) for v in archive.read("capture-results.ndjson").decode().splitlines()]
    walk = diagnostic["walkStartedBootNs"]
    report = {"sourceSha256": hashlib.file_digest(trial_zip.open("rb"), "sha256").hexdigest(),
        "referenceRouteMetresOperatorReported": None, "independentRouteMeasurementVerified": False,
        "captureSeconds": (manifest["endedBootNs"]-manifest["startedBootNs"])/1e9,
        "movementTriggerSecondsAfterCapture": (walk-manifest["startedBootNs"])/1e9,
        "movementTriggerToFinishSeconds": (manifest["endedBootNs"]-walk)/1e9,
        "profileIncluded": "research-profile.json" in archive.namelist(),
        "appRelease": manifest["appRelease"], "onDeviceStatus": diagnostic["status"],
        "onDeviceReason": diagnostic["reason"], "sensorStats": {}}
    for sensor in ("accelerometer", "gyroscope", "magnetometer"):
        rows = [v for v in imu if v["sensor"] == sensor]
        t = np.array([v["timestampNs"] for v in rows], dtype=np.int64)
        values = np.array([[v[a] for a in "xyz"] for v in rows])
        gaps = np.diff(t)/1e6
        report["sensorStats"][sensor] = {"count": len(rows),
            "meanHz": (len(t)-1)/((t[-1]-t[0])/1e9), "maxGapMs": float(gaps.max()),
            "p99GapMs": float(np.percentile(gaps,99)), "strictlyMonotonic": bool(np.all(gaps>0)),
            "allFinite": bool(np.isfinite(values).all()), "walkingPhaseCount": int(np.sum(t>=walk))}
    t = np.array([v["timestampNs"] for v in frames], dtype=np.int64)
    camera = {v["timestampNs"]: v for v in camera_results}
    matched = [camera[v["timestampNs"]] for v in frames if v["timestampNs"] in camera]
    report["camera"] = {"frames": len(frames), "meanStoredHz": (len(t)-1)/((t[-1]-t[0])/1e9),
        "maxGapMs": float(np.diff(t).max()/1e6), "matchedCaptureResults": len(matched),
        "medianArrivalDelayMs": float(np.median([(v["arrivalBootNs"]-v["timestampNs"])/1e6 for v in frames])),
        "exposureRangeMs": [min(v["exposureNs"] for v in matched)/1e6, max(v["exposureNs"] for v in matched)/1e6],
        "readoutRangeMs": [min(v["rollingShutterSkewNs"] for v in matched)/1e6, max(v["rollingShutterSkewNs"] for v in matched)/1e6]}
    motion = []
    for second in range(-2, int(np.ceil(report["movementTriggerToFinishSeconds"]))):
        row = {"secondsFromMovementTrigger": second}
        for sensor in ("accelerometer", "gyroscope"):
            values = np.array([[v[a] for a in "xyz"] for v in imu if v["sensor"]==sensor and second<=(v["timestampNs"]-walk)/1e9<second+1])
            if len(values):
                norm = np.linalg.norm(values,axis=1)
                row[sensor+"NormStd"] = float(np.std(norm))
                row[sensor+"NormMean"] = float(np.mean(norm))
        motion.append(row)
    report["oneSecondMotionWindows"] = motion
    sheet = Image.new("RGB",(960,810),(245,245,245)); draw = ImageDraw.Draw(sheet)
    for j,relative in enumerate([-1,0,.8,1.6,2.4,3.2,5,8,11]):
        frame = min(frames,key=lambda v:abs((v["timestampNs"]-walk)/1e9-relative))
        bgr = cv2.imdecode(np.frombuffer(archive.read(frame["file"]),np.uint8),cv2.IMREAD_COLOR)
        image = Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)); image.thumbnail((320,240))
        x=(j%3)*320; y=(j//3)*270; sheet.paste(image,(x,y+25))
        draw.text((x+6,y+7),f'{(frame["timestampNs"]-walk)/1e9:.2f}s / {frame["file"]}',fill="black")
    sheet.save(trial_out/"walk-contact-sheet.jpg")
(trial_out/"capture-audit.json").write_text(json.dumps(report,indent=2))
print(json.dumps({k:v for k,v in report.items() if k!="oneSecondMotionWindows"},indent=2))


{
  "sourceSha256": "e53a543aa1211aeb5042c2fc5d76355f6f1a3f466f842a11ee2398a304068899",
  "referenceRouteMetresOperatorReported": null,
  "independentRouteMeasurementVerified": false,
  "captureSeconds": 29.649163374,
  "movementTriggerSecondsAfterCapture": 16.672774015,
  "movementTriggerToFinishSeconds": 12.976389359,
  "profileIncluded": true,
  "appRelease": {
    "appVersion": "0.3.4",
    "buildNumber": "11",
    "runtimeVersion": "36ef802b5c8463d2c0a172abd85bb56d75f0ea0e",
    "updateId": "ac8ee83a-60a4-47e6-9ae1-9d2b48abdc59",
    "channel": "preview",
    "embedded": true
  },
  "onDeviceStatus": "rejected",
  "onDeviceReason": "reprojection-error",
  "sensorStats": {
    "accelerometer": {
      "count": 6179,
      "meanHz": 210.36828799607298,
      "maxGapMs": 4.922291,
      "p99GapMs": 4.859532,
      "strictlyMonotonic": true,
      "allFinite": true,
      "walkingPhaseCount": 2670
    },
    "gyroscope": {
      "count": 6178,
      "meanHz": 210.36858483275117,
     

In [33]:
# Explicit JPEG artifact replay, beginning at the first native tracking-log timestamp.
# JPEG decoding and desktop JNI can differ from native luma; device results remain primary.
TOOLS=ROOT/"android/dist/native-distance-tools"
JDK=next(v for v in sorted(TOOLS.glob("jdk*")) if v.is_dir() and (v/"bin/javac.exe").exists())
classpath=os.pathsep.join(str(TOOLS/v) for v in ["opencv.jar","commons-math3.jar","json.jar"])
classes=TOOLS/"test-classes";tests=ROOT/"android/research/native-tests"
source=ROOT/"android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/ResearchDistanceTracker.java"
subprocess.run([str(JDK/"bin/javac.exe"),"-cp",classpath,"-d",str(classes),str(tests/"OpenCVLoader.java"),str(source),str(tests/"NativeDistanceAlgorithmCheck.java"),str(tests/"NativeDistanceCaptureReplay.java")],check=True,capture_output=True,text=True)
with zipfile.ZipFile(trial_zip) as archive:
    profile=json.loads(archive.read("research-profile.json"))
    for key in ["installationId","cameraId","width","height","imageAxes","sensorOrientationDegrees","stabilizationRequested"]:
        assert profile["binding"][key]==manifest[key], key
    profile_file=trial_out/"capture-research-profile.json"
    profile_file.write_text(json.dumps(profile))
    native_rows=[json.loads(v) for v in archive.read("tracking.ndjson").decode().splitlines()]
    first_loss=next(v for v in native_rows if v["tracking"]["state"]=="tracking-lost")
    native_summary={"onDeviceStatus":diagnostic["status"],"onDeviceReason":diagnostic["reason"],
        "onDevicePoseFrames":diagnostic["poseFrames"],"mapPointsAdded":diagnostic["mapPointsAdded"],
        "keyframes":diagnostic["keyframes"],"failureSecondsAfterMovementTrigger":(first_loss["timestampNs"]-walk)/1e9,
        "reprojectionRmsPixels":diagnostic["reprojectionRmsPixels"],"currentReprojectionGatePixels":1.5,
        "firstTrackingFrameOffsetSeconds":(native_rows[0]["timestampNs"]-walk)/1e9,
        "validMetricDistanceProduced":False}
command=[str(JDK/"bin/java.exe"),"-Djava.library.path="+str(TOOLS/"windows-opencv"),"-cp",str(classes)+os.pathsep+classpath]
replay_file=trial_out/"native-timestamp-jpeg-replay.json"
result=subprocess.run(command+["NativeDistanceCaptureReplay",str(TOOLS/"windows-opencv/opencv_java4120.dll"),str(trial_zip),str(profile_file),str(replay_file)],check=True,capture_output=True,text=True)
r=json.loads(replay_file.read_text())
summary={k:r.get(k) for k in ["status","reason","poseFrames","replayedFrames","features","reprojectionRmsPixels","mapPointsAdded","keyframes","replayFirstFrameBootNs","replayStartsFromNativeTrackingLog"]}
summary["allImuAvailableBeforeFrames"]=True
summary["algorithmSourceSha256"]=hashlib.sha256(source.read_bytes()).hexdigest()
(trial_out/"native-vs-replay-summary.json").write_text(json.dumps({"actualNativeTrial":native_summary,"explicitJpegArtifactReplay":summary},indent=2))
print(json.dumps({"actualNativeTrial":native_summary,"explicitJpegArtifactReplay":summary},indent=2))


{
  "actualNativeTrial": {
    "onDeviceStatus": "rejected",
    "onDeviceReason": "reprojection-error",
    "onDevicePoseFrames": 1,
    "mapPointsAdded": 0,
    "keyframes": 0,
    "failureSecondsAfterMovementTrigger": 0.230164597,
    "reprojectionRmsPixels": 1.6477894284612442,
    "currentReprojectionGatePixels": 1.5,
    "firstTrackingFrameOffsetSeconds": -0.002966705,
    "validMetricDistanceProduced": false
  },
  "explicitJpegArtifactReplay": {
    "status": "rejected",
    "reason": "reprojection-error",
    "poseFrames": 1,
    "replayedFrames": 161,
    "features": 136,
    "reprojectionRmsPixels": 1.615179944064254,
    "mapPointsAdded": 0,
    "keyframes": 1,
    "replayFirstFrameBootNs": 22728718346378,
    "replayStartsFromNativeTrackingLog": true,
    "allImuAvailableBeforeFrames": true,
    "algorithmSourceSha256": "db8082c1de30f17a5687dd204c92f1ec81dda6bfc68b0f31177ddd553ca44b4b"
  }
}


### Latest physical retry - build 11, profile present

Reviewed distance-trial-1791349028491 (SHA-256
e53a543aa1211aeb5042c2fc5d76355f6f1a3f466f842a11ee2398a304068899).
Actual 0.3.4/build 11 native capture contains the matching research profile.
On-device tracking obtains one pose, then rejects reprojection-error 0.230 s
after the movement trigger: coordinate reprojection RMS 1.6478 px exceeds the
unchanged 1.5 px engineering gate. It adds zero landmarks and zero keyframes.
The map-renewal stage therefore never becomes useful on this live trial. No
metric estimate, distance error, gait speed or independent validation is available.
Route length was not restated, so it is not assumed to be 3 m here.

Acquisition is healthy: 367 camera frames at 12.64 Hz, all with capture-result
metadata, max frame gap 133.2 ms. Acceleration 6,179 and gyro 6,178 samples at
210.37 Hz, max gaps 4.92 ms, magnetic 1,469 at 50 Hz, max gap 20 ms. All sensor
values finite and times strictly monotonic. Automatic quiet-stop completed the
29.65 s capture, 12.98 s after the movement trigger. Finish includes quiet time,
so this duration is not walking speed or a measured-course timing result.

Explicit JPEG replay, supplied the profile stored in this ZIP, also produces one
pose and rejects reprojection-error (1.6152 px). Its feature count differs from
native luma, so it is not a bit-identical live replay. The reusable desktop
replayer now starts at the actual first tracking-log camera timestamp instead
of dropping frames just before the wall/motion trigger. First native frame is
2.97 ms before that trigger in this capture. Earlier replay outputs are retained
with their original cutoff semantics and are not relabelled as device results.
All IMU samples are available before images offline, so live latency is untested.

Sampled images show holder obstruction, walking blur and textured room features.
Their individual causal contributions have not been isolated. Current evidence
supports a fragile visual-map initialization, not missing profile or sensor data.
Next work should validate initialization over multiple views and investigate
holder-feature rejection/rolling-readout and blur handling. Do not relax the
1.5 px gate or request repeated walks as if profile import solved odometry.
The earlier retained walk's 151-pose replay did not demonstrate general live
robustness, and its scale also remained rejected. Raw ZIP/profile/images and
outputs stay ignored. Actual audit/replay code and results are in the existing
notebook. No Android source change, APK, OTA or Git push in this review.



## Better-placement physical retry - distance-trial-1791349358636

User reports the preceding trial involved a slip during setup and this trial had better placement. This is operator context, not an independently timed slip event. Route length was not restated. Native results remain primary and the recorded profile is reused only for explicit JPEG replay.


In [34]:
# New execution on the first uploaded measured-route artifact, not a live capture.
from pathlib import Path
import json, zipfile, hashlib, os, subprocess
import numpy as np
import cv2
from PIL import Image, ImageDraw
ROOT = Path.cwd() if (Path.cwd()/"android").exists() else Path.cwd().parent
trial_zip = Path(r"C:/Users/frank/Downloads/Telegram Desktop/distance-trial-1791349358636.zip")
trial_out = ROOT/"android/dist/camera-distance/distance-trial-1791349358636"
trial_out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(trial_zip) as archive:
    assert archive.testzip() is None
    manifest = json.loads(archive.read("manifest.json"))
    diagnostic = json.loads(archive.read("distance-diagnostics.json"))
    frames = [json.loads(v) for v in archive.read("frames.ndjson").decode().splitlines()]
    imu = [json.loads(v) for v in archive.read("imu.ndjson").decode().splitlines()]
    camera_results = [json.loads(v) for v in archive.read("capture-results.ndjson").decode().splitlines()]
    walk = diagnostic["walkStartedBootNs"]
    report = {"sourceSha256": hashlib.file_digest(trial_zip.open("rb"), "sha256").hexdigest(),
        "referenceRouteMetresOperatorReported": None, "independentRouteMeasurementVerified": False,
        "captureSeconds": (manifest["endedBootNs"]-manifest["startedBootNs"])/1e9,
        "movementTriggerSecondsAfterCapture": (walk-manifest["startedBootNs"])/1e9,
        "movementTriggerToFinishSeconds": (manifest["endedBootNs"]-walk)/1e9,
        "profileIncluded": "research-profile.json" in archive.namelist(),
        "appRelease": manifest["appRelease"], "onDeviceStatus": diagnostic["status"],
        "onDeviceReason": diagnostic["reason"], "sensorStats": {}}
    for sensor in ("accelerometer", "gyroscope", "magnetometer"):
        rows = [v for v in imu if v["sensor"] == sensor]
        t = np.array([v["timestampNs"] for v in rows], dtype=np.int64)
        values = np.array([[v[a] for a in "xyz"] for v in rows])
        gaps = np.diff(t)/1e6
        report["sensorStats"][sensor] = {"count": len(rows),
            "meanHz": (len(t)-1)/((t[-1]-t[0])/1e9), "maxGapMs": float(gaps.max()),
            "p99GapMs": float(np.percentile(gaps,99)), "strictlyMonotonic": bool(np.all(gaps>0)),
            "allFinite": bool(np.isfinite(values).all()), "walkingPhaseCount": int(np.sum(t>=walk))}
    t = np.array([v["timestampNs"] for v in frames], dtype=np.int64)
    camera = {v["timestampNs"]: v for v in camera_results}
    matched = [camera[v["timestampNs"]] for v in frames if v["timestampNs"] in camera]
    report["camera"] = {"frames": len(frames), "meanStoredHz": (len(t)-1)/((t[-1]-t[0])/1e9),
        "maxGapMs": float(np.diff(t).max()/1e6), "matchedCaptureResults": len(matched),
        "medianArrivalDelayMs": float(np.median([(v["arrivalBootNs"]-v["timestampNs"])/1e6 for v in frames])),
        "exposureRangeMs": [min(v["exposureNs"] for v in matched)/1e6, max(v["exposureNs"] for v in matched)/1e6],
        "readoutRangeMs": [min(v["rollingShutterSkewNs"] for v in matched)/1e6, max(v["rollingShutterSkewNs"] for v in matched)/1e6]}
    motion = []
    for second in range(-2, int(np.ceil(report["movementTriggerToFinishSeconds"]))):
        row = {"secondsFromMovementTrigger": second}
        for sensor in ("accelerometer", "gyroscope"):
            values = np.array([[v[a] for a in "xyz"] for v in imu if v["sensor"]==sensor and second<=(v["timestampNs"]-walk)/1e9<second+1])
            if len(values):
                norm = np.linalg.norm(values,axis=1)
                row[sensor+"NormStd"] = float(np.std(norm))
                row[sensor+"NormMean"] = float(np.mean(norm))
        motion.append(row)
    report["oneSecondMotionWindows"] = motion
    sheet = Image.new("RGB",(960,810),(245,245,245)); draw = ImageDraw.Draw(sheet)
    for j,relative in enumerate([-1,0,.8,1.6,2.4,3.2,5,8,11]):
        frame = min(frames,key=lambda v:abs((v["timestampNs"]-walk)/1e9-relative))
        bgr = cv2.imdecode(np.frombuffer(archive.read(frame["file"]),np.uint8),cv2.IMREAD_COLOR)
        image = Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)); image.thumbnail((320,240))
        x=(j%3)*320; y=(j//3)*270; sheet.paste(image,(x,y+25))
        draw.text((x+6,y+7),f'{(frame["timestampNs"]-walk)/1e9:.2f}s / {frame["file"]}',fill="black")
    sheet.save(trial_out/"walk-contact-sheet.jpg")
(trial_out/"capture-audit.json").write_text(json.dumps(report,indent=2))
print(json.dumps({k:v for k,v in report.items() if k!="oneSecondMotionWindows"},indent=2))


{
  "sourceSha256": "b47807a3ccfc7f0d1ac57830268b88c0d137b7883b0297a0803aea1490bb1642",
  "referenceRouteMetresOperatorReported": null,
  "independentRouteMeasurementVerified": false,
  "captureSeconds": 28.478364833,
  "movementTriggerSecondsAfterCapture": 13.67731538,
  "movementTriggerToFinishSeconds": 14.801049453,
  "profileIncluded": true,
  "appRelease": {
    "appVersion": "0.3.4",
    "buildNumber": "11",
    "runtimeVersion": "36ef802b5c8463d2c0a172abd85bb56d75f0ea0e",
    "updateId": "ac8ee83a-60a4-47e6-9ae1-9d2b48abdc59",
    "channel": "preview",
    "embedded": true
  },
  "onDeviceStatus": "rejected",
  "onDeviceReason": "reprojection-error",
  "sensorStats": {
    "accelerometer": {
      "count": 5938,
      "meanHz": 210.40671327423092,
      "maxGapMs": 4.891875,
      "p99GapMs": 4.84624132,
      "strictlyMonotonic": true,
      "allFinite": true,
      "walkingPhaseCount": 3058
    },
    "gyroscope": {
      "count": 5938,
      "meanHz": 210.40671327423092,
    

In [35]:
# Explicit JPEG artifact replay, beginning at the first native tracking-log timestamp.
# JPEG decoding and desktop JNI can differ from native luma; device results remain primary.
TOOLS=ROOT/"android/dist/native-distance-tools"
JDK=next(v for v in sorted(TOOLS.glob("jdk*")) if v.is_dir() and (v/"bin/javac.exe").exists())
classpath=os.pathsep.join(str(TOOLS/v) for v in ["opencv.jar","commons-math3.jar","json.jar"])
classes=TOOLS/"test-classes";tests=ROOT/"android/research/native-tests"
source=ROOT/"android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/ResearchDistanceTracker.java"
subprocess.run([str(JDK/"bin/javac.exe"),"-cp",classpath,"-d",str(classes),str(tests/"OpenCVLoader.java"),str(source),str(tests/"NativeDistanceAlgorithmCheck.java"),str(tests/"NativeDistanceCaptureReplay.java")],check=True,capture_output=True,text=True)
with zipfile.ZipFile(trial_zip) as archive:
    profile=json.loads(archive.read("research-profile.json"))
    for key in ["installationId","cameraId","width","height","imageAxes","sensorOrientationDegrees","stabilizationRequested"]:
        assert profile["binding"][key]==manifest[key], key
    profile_file=trial_out/"capture-research-profile.json"
    profile_file.write_text(json.dumps(profile))
    native_rows=[json.loads(v) for v in archive.read("tracking.ndjson").decode().splitlines()]
    first_loss=next(v for v in native_rows if v["tracking"]["state"]=="tracking-lost")
    native_summary={"onDeviceStatus":diagnostic["status"],"onDeviceReason":diagnostic["reason"],
        "onDevicePoseFrames":diagnostic["poseFrames"],"mapPointsAdded":diagnostic["mapPointsAdded"],
        "keyframes":diagnostic["keyframes"],"failureSecondsAfterMovementTrigger":(first_loss["timestampNs"]-walk)/1e9,
        "reprojectionRmsPixels":diagnostic["reprojectionRmsPixels"],"currentReprojectionGatePixels":1.5,
        "firstTrackingFrameOffsetSeconds":(native_rows[0]["timestampNs"]-walk)/1e9,
        "validMetricDistanceProduced":False}
command=[str(JDK/"bin/java.exe"),"-Djava.library.path="+str(TOOLS/"windows-opencv"),"-cp",str(classes)+os.pathsep+classpath]
replay_file=trial_out/"native-timestamp-jpeg-replay.json"
result=subprocess.run(command+["NativeDistanceCaptureReplay",str(TOOLS/"windows-opencv/opencv_java4120.dll"),str(trial_zip),str(profile_file),str(replay_file)],check=True,capture_output=True,text=True)
r=json.loads(replay_file.read_text())
summary={k:r.get(k) for k in ["status","reason","poseFrames","replayedFrames","features","reprojectionRmsPixels","mapPointsAdded","keyframes","replayFirstFrameBootNs","replayStartsFromNativeTrackingLog"]}
summary["allImuAvailableBeforeFrames"]=True
summary["algorithmSourceSha256"]=hashlib.sha256(source.read_bytes()).hexdigest()
(trial_out/"native-vs-replay-summary.json").write_text(json.dumps({"actualNativeTrial":native_summary,"explicitJpegArtifactReplay":summary},indent=2))
print(json.dumps({"actualNativeTrial":native_summary,"explicitJpegArtifactReplay":summary},indent=2))


{
  "actualNativeTrial": {
    "onDeviceStatus": "rejected",
    "onDeviceReason": "reprojection-error",
    "onDevicePoseFrames": 6,
    "mapPointsAdded": 25,
    "keyframes": 1,
    "failureSecondsAfterMovementTrigger": 0.721140941,
    "reprojectionRmsPixels": 1.5536998042091754,
    "currentReprojectionGatePixels": 1.5,
    "firstTrackingFrameOffsetSeconds": -0.011544111,
    "validMetricDistanceProduced": false
  },
  "explicitJpegArtifactReplay": {
    "status": "rejected",
    "reason": "reprojection-error",
    "poseFrames": 1,
    "replayedFrames": 185,
    "features": 43,
    "reprojectionRmsPixels": 1.7598429530128405,
    "mapPointsAdded": 0,
    "keyframes": 1,
    "replayFirstFrameBootNs": 23055383626245,
    "replayStartsFromNativeTrackingLog": true,
    "allImuAvailableBeforeFrames": true,
    "algorithmSourceSha256": "db8082c1de30f17a5687dd204c92f1ec81dda6bfc68b0f31177ddd553ca44b4b"
  }
}


### Better-placement physical trial - 7 October 2026

User reports a slip during the preceding setup and better mounting in the new
trial distance-trial-1791349358636. Source SHA-256:
b47807a3ccfc7f0d1ac57830268b88c0d137b7883b0297a0803aea1490bb1642.
Profile is present and release remains 0.3.4/build 11. Native tracking improves
from one to six poses, with 25 renewed map points and one candidate keyframe.
Failure occurs 0.721 s after the movement trigger instead of 0.230 s. Both runs
reject reprojection-error: 1.5537 px here, previously 1.6478 px, versus the same
1.5 px engineering gate. No accepted scale, metre estimate or route error.
This is modest tracking improvement, not a successful distance result.

Full capture lasts 28.478 s; visual tracking begins around the movement trigger
13.677 s after capture starts. Setup duration alone does not enter the visual
trajectory; early IMU rows are stored but scale interpolation would use pose
segment times. There is appreciable movement in the two seconds preceding the
trigger. The records do not identify an exact slip time or prove that motion
was placement rather than walking, and do not validate first-step timing.
The operator report makes a placement effect plausible, not an isolated causal
finding. Failure with better placement shows handling is not the whole problem.

Acquisition remains healthy: 353 frames at 12.686 Hz, all matched to camera
metadata, max gap 103.6 ms. Accel/gyro 5,938 each at 210.407 Hz with max gap
4.89 ms, magnetometer 1,411 at 50 Hz with max gap 20.0 ms. All finite/monotonic.
JPEG artifact replay using the included profile and actual first native frame
rejects after one pose (1.7598 px). JPEG/native luma and RANSAC differences mean
it is not bit-identical and cannot overwrite the actual six-pose device result.
Sampled images still show holder obstruction and walking blur. No isolated cause
or validated distance error can be inferred. Route length was not restated.

Both profile-loaded physical captures remain available to validate a more robust
bootstrap rather than fitting only the first walk. Next work remains multi-view
initialization, camera-motion/occlusion handling and start-cue timing, followed
by metric consistency. Do not relax the gate just to accept the near-threshold
sample. Executed audit/replay outputs are in the existing notebook. Raw/private
artifacts stay ignored. No app change, new APK, OTA or source publication in this
review, and the existing publication scope has not been approved.



## Established VIO reference feasibility ? 7 October 2026

New execution, not a replay of earlier estimator results. Pinned upstream VINS-Mono
90dabb5ec79946ae42fd2e1e91d4e69aabe1e25d and Ceres 1.14.0
facb199f3eda902360f9e1d5271372b7e54febe1. An offline wrapper compiles unchanged
upstream estimator sources with minimal logging/header shims, replacing ROS transport.
The diagnostic Python LK/forward-backward/calibrated-epipolar frontend is **not** the
unchanged VINS frontend. All five lens coefficients and raw pixel rows are preserved.
No benchmark claim against the full original ROS pipeline is made.

Inputs use actual hardware camera times plus exposure/readout midpoint. The measured
research time offset is applied once by the estimator. Asynchronous acceleration is
interpolated at actual gyro times only between valid bracketing samples; no camera
upsampling. Rotation is measured; camera-to-IMU translation starts from an explicit
zero guess with online refinement. Noise uses disclosed EuRoC reference placeholders,
not measured phone statistics. JPEG decoding differs from live native luma.
Results are diagnostic only, with distanceReady/fullCalibrationReady false.


In [36]:
from pathlib import Path
import sys, json
ROOT=Path.cwd()
sys.path.insert(0,str(ROOT/'android/research'))
from vio_reference import export_capture, export_feature_tracks, write_reference_config
base=ROOT/'android/dist/vio-benchmark'
profile=ROOT/'android/dist/camera-calibration/calibration-1791298033545/GaitTrace-phone-research-profile.json'
reports=[]
for ident in ['1791347211905','1791349028491','1791349358636']:
    source=Path('C:/Users/frank/Downloads/Telegram Desktop')/f'distance-trial-{ident}.zip'
    dest=base/f'distance-trial-{ident}'
    report=export_capture(source,dest,profile)
    report['frontend']=export_feature_tracks(dest)
    report['assumptions']=write_reference_config(dest)
    reports.append(report)
    print(json.dumps({k:report[k] for k in ['captureId','frameCount','cameraHz','accelHz','gyroHz','pairedImuCount','frontend']},indent=2),flush=True)
(base/'input-reports.json').write_text(json.dumps(reports,indent=2))


{
  "captureId": "distance-trial-1791347211905",
  "frameCount": 270,
  "cameraHz": 12.81933725270594,
  "accelHz": 210.40092452056996,
  "gyroHz": 210.40092452056996,
  "pairedImuCount": 4509,
  "frontend": {
    "frames": 270,
    "minFeatures": 150,
    "medianFeatures": 150.0,
    "frontend": "diagnostic-python-lk-fb-calibrated-f-v1",
    "fullCalibrationReady": false,
    "distanceReady": false,
    "featuresSha256": "02d3bc3d6ecd3f1314f44e219df3cfcaa2471a594648ba9632c10743fe819747"
  }
}
{
  "captureId": "distance-trial-1791349028491",
  "frameCount": 367,
  "cameraHz": 12.645011572306737,
  "accelHz": 210.36828799607298,
  "gyroHz": 210.36858483275117,
  "pairedImuCount": 6177,
  "frontend": {
    "frames": 367,
    "minFeatures": 150,
    "medianFeatures": 150.0,
    "frontend": "diagnostic-python-lk-fb-calibrated-f-v1",
    "fullCalibrationReady": false,
    "distanceReady": false,
    "featuresSha256": "771339d1567e235976dcf060c4f5de76905c43d7ea91a72a4f08d21783f4d707"
  }
}
{

In [37]:
from pathlib import Path
import subprocess, json, csv, hashlib
ROOT=Path.cwd();base=ROOT/'android/dist/vio-benchmark'
engine='/root/gaittrace-vio/reference-build/vins_reference'
source_info=json.loads((base/'VINS-Mono-source.json').read_text())
results=[]
for ident in ['1791347211905','1791349028491','1791349358636']:
    directory=base/f'distance-trial-{ident}'
    linux='/mnt/c/'+directory.as_posix()[3:]
    r=subprocess.run(['wsl','-d','Ubuntu','--',engine,linux],text=True,capture_output=True)
    (directory/'reference-stdout.txt').write_text(r.stdout)
    (directory/'reference-stderr.txt').write_text(r.stderr)
    with (directory/'reference-states.csv').open() as f: rows=list(csv.DictReader(f))
    ready=[x for x in rows if x['initialized']=='1']
    result={'captureId':f'distance-trial-{ident}','exitCode':r.returncode,
            'framesProcessed':len(rows),'initializedFrames':len(ready),
            'firstInitializedCameraTimeSeconds':float(ready[0]['camera_time_s']) if ready else None,
            'initializationMessages':r.stderr.count('Initialization finish!'),
            'rebootMessages':r.stderr.count('system reboot!'),
            'insufficientExcitationMessages':r.stderr.count('IMU excitation not enouth!'),
            'insufficientParallaxMessages':r.stderr.count('Not enough features or parallax'),
            'upstreamCommit':source_info['commit'],'distanceReady':False,'fullCalibrationReady':False,
            'scope':'New offline estimator feasibility execution; diagnostic frontend and provisional noise/translation, not physical distance accuracy'}
    results.append(result)
    (directory/'reference-summary.json').write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2),flush=True)
(base/'reference-results.json').write_text(json.dumps(results,indent=2))


{
  "captureId": "distance-trial-1791347211905",
  "exitCode": 0,
  "framesProcessed": 270,
  "initializedFrames": 254,
  "firstInitializedCameraTimeSeconds": 20896.843033873,
  "initializationMessages": 1,
  "rebootMessages": 0,
  "insufficientExcitationMessages": 3,
  "insufficientParallaxMessages": 3,
  "upstreamCommit": "90dabb5ec79946ae42fd2e1e91d4e69aabe1e25d",
  "distanceReady": false,
  "fullCalibrationReady": false,
  "scope": "New offline estimator feasibility execution; diagnostic frontend and provisional noise/translation, not physical distance accuracy"
}
{
  "captureId": "distance-trial-1791349028491",
  "exitCode": 0,
  "framesProcessed": 367,
  "initializedFrames": 263,
  "firstInitializedCameraTimeSeconds": 22720.689735124,
  "initializationMessages": 1,
  "rebootMessages": 0,
  "insufficientExcitationMessages": 0,
  "insufficientParallaxMessages": 44,
  "upstreamCommit": "90dabb5ec79946ae42fd2e1e91d4e69aabe1e25d",
  "distanceReady": false,
  "fullCalibrationReady": fa

In [38]:
from pathlib import Path
import sys, subprocess, json
ROOT=Path.cwd();sys.path.insert(0,str(ROOT/'android/research'))
from vio_reference_fixture import create_control,evaluate_control
p=ROOT/'android/dist/vio-benchmark/analytic-control'
create_control(p)
linux='/mnt/c/'+p.as_posix()[3:]
r=subprocess.run(['wsl','-d','Ubuntu','--','/root/gaittrace-vio/reference-build/vins_reference',linux],text=True,capture_output=True)
(p/'reference-stdout.txt').write_text(r.stdout);(p/'reference-stderr.txt').write_text(r.stderr)
assert r.returncode==0,r.stderr
result=evaluate_control(p)
(p/'control-summary.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "kind": "analytic-nongait-harness-control",
  "initializedFrames": 559,
  "rigidOnlyPositionRmseMetres": 0.0003601995187483881,
  "similarityScaleFitted": false,
  "passed": true,
  "independentDistanceValidation": false,
  "distanceReady": false
}


### Harness initialization and geometry integrity follow-up

The upstream ROS estimator has static storage. The wrapper now explicitly matches
that zero-initialization contract, rather than relying on local-stack pointer
contents before the upstream constructor. Earlier replay/control outputs are
retained as preliminary wrapper executions; this follow-up is authoritative for
the corrected harness. Added calibration diagnostics do not change estimator math.
This rerun resolves the specific outstanding gap: plausible-looking position
outputs could conceal impossible camera/IMU geometry. All outputs use a separate
`verified` prefix, preserving the earlier artifacts.


In [39]:
from pathlib import Path
import sys,subprocess,json,hashlib
ROOT=Path.cwd();sys.path.insert(0,str(ROOT/'android/research'))
from vio_reference import summarize_reference
from vio_reference_fixture import evaluate_control
base=ROOT/'android/dist/vio-benchmark';prefix='verified'
engine='/root/gaittrace-vio/reference-build/vins_reference'
def replay(directory):
    linux='/mnt/c/'+directory.as_posix()[3:]
    result=subprocess.run(['wsl','-d','Ubuntu','--',engine,linux,prefix],text=True,capture_output=True)
    (directory/(prefix+'-stdout.txt')).write_text(result.stdout)
    (directory/(prefix+'-stderr.txt')).write_text(result.stderr)
    assert result.returncode==0,result.stderr
replay(base/'analytic-control')
control=evaluate_control(base/'analytic-control',prefix)
(base/'analytic-control/verified-control-summary.json').write_text(json.dumps(control,indent=2))
print('Corrected harness control:',json.dumps(control),flush=True)
results=[]
for ident in ['1791347211905','1791349028491','1791349358636']:
    directory=base/f'distance-trial-{ident}'
    replay(directory)
    report=summarize_reference(directory,Path('C:/Users/frank/Downloads/Telegram Desktop')/(directory.name+'.zip'),prefix)
    results.append(report)
    print(json.dumps(report,indent=2),flush=True)
(base/'verified-integrity-results.json').write_text(json.dumps(results,indent=2))


Corrected harness control: {"kind": "analytic-nongait-harness-control", "initializedFrames": 559, "rigidOnlyPositionRmseMetres": 0.0003601995187483881, "similarityScaleFitted": false, "passed": true, "independentDistanceValidation": false, "distanceReady": false}
{
  "captureId": "distance-trial-1791347211905",
  "initializedFrames": 254,
  "initializedFramesAfterAppTrigger": 154,
  "lastFrameInitialized": true,
  "reboots": 0,
  "maxInitializedStateSpeedMetresPerSecond": 0.700055262587925,
  "maxInitializedAccelerationBiasNorm": 0.34062957304540065,
  "maxInitializedGyroBiasNorm": 0.018440664144848882,
  "initialTimeOffsetSeconds": -0.029728361132368618,
  "finalTimeOffsetSeconds": -0.003630948006217021,
  "distanceReady": false,
  "fullCalibrationReady": false,
  "independentDistanceValidation": false,
  "limitations": [
    "App movement trigger is not verified first step/course boundary",
    "IMU position state is not an accepted complete walking-distance measurement",
    "Online

In [40]:
from pathlib import Path
import sys,subprocess,json,hashlib
ROOT=Path.cwd();base=ROOT/'android/dist/vio-benchmark';sys.path.insert(0,str(ROOT/'android/research'))
from vio_reference import summarize_reference
reports=[]
for ident in ['1791347211905','1791349028491','1791349358636']:
    directory=base/f'distance-trial-{ident}'
    report=summarize_reference(directory,Path('C:/Users/frank/Downloads/Telegram Desktop')/(directory.name+'.zip'),'verified')
    reports.append(report)
    print(json.dumps({k:report[k] for k in ['captureId','initializedFrames','reboots','lastFrameInitialized',
      'maxInitializedCameraImuTranslationNormMetres','maxCameraImuTranslationNormAfterAppTriggerMetres',
      'lastInitializedCameraImuTranslationNormMetres','maxInitializedRotationChangeDegrees',
      'provisionalImuPositionPathAfterAppTriggerMetres','distanceReady']},indent=2))
(base/'verified-integrity-results.json').write_text(json.dumps(reports,indent=2))
checks=subprocess.run(['C:/Users/frank/.venv/Scripts/python.exe','-m','unittest','discover','-s','android/research','-p','test_*.py'],text=True,capture_output=True)
assert checks.returncode==0,checks.stderr
provenance={'date':'2026-10-07','kind':'new-offline-reference-execution',
  'vinsCommit':'90dabb5ec79946ae42fd2e1e91d4e69aabe1e25d',
  'ceresCommit':'facb199f3eda902360f9e1d5271372b7e54febe1',
  'openVinsCommit':'69488123ed9362dd44b6f28e7f4680abbff1442b',
  'openVinsScope':'Library compiled only; no phone replay or Android integration',
  'pythonTests':checks.stderr.strip(),
  'sourceHashes':{str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest()
    for f in [ROOT/'android/research/vio_reference.py',ROOT/'android/research/vio_reference_fixture.py',ROOT/'android/research/test_vio_reference.py',
              *(ROOT/'android/research/vio-reference').rglob('*')] if f.is_file()},
  'vinsBinarySha256':subprocess.check_output(['wsl','-d','Ubuntu','--','sha256sum','/root/gaittrace-vio/reference-build/vins_reference'],text=True).split()[0],
  'openVinsLibrarySha256':subprocess.check_output(['wsl','-d','Ubuntu','--','sha256sum','/root/gaittrace-vio/openvins-build/libov_msckf_lib.so'],text=True).split()[0],
  'vinsSourceWorkingTree':subprocess.check_output(['wsl','-d','Ubuntu','--','git','-C','/root/gaittrace-vio/vins-mono','status','--short'],text=True),
  'openVinsCompatibilityDiff':subprocess.check_output(['wsl','-d','Ubuntu','--','git','-C','/root/gaittrace-vio/open-vins','diff'],text=True),
  'distanceReady':False,'fullCalibrationReady':False,'independentDistanceValidation':False,
  'newApkOrAppIntegration':False}
(base/'execution-provenance.json').write_text(json.dumps(provenance,indent=2))
print(json.dumps(provenance,indent=2))


{
  "captureId": "distance-trial-1791347211905",
  "initializedFrames": 254,
  "reboots": 0,
  "lastFrameInitialized": true,
  "maxInitializedCameraImuTranslationNormMetres": 8.106877207277611,
  "maxCameraImuTranslationNormAfterAppTriggerMetres": 0.1291814612057032,
  "lastInitializedCameraImuTranslationNormMetres": 0.12529194282667586,
  "maxInitializedRotationChangeDegrees": 27.35885818760896,
  "provisionalImuPositionPathAfterAppTriggerMetres": 3.1223584862932885,
  "distanceReady": false
}
{
  "captureId": "distance-trial-1791349028491",
  "initializedFrames": 263,
  "reboots": 0,
  "lastFrameInitialized": true,
  "maxInitializedCameraImuTranslationNormMetres": 0.6592610133451576,
  "maxCameraImuTranslationNormAfterAppTriggerMetres": 0.1390873004870477,
  "lastInitializedCameraImuTranslationNormMetres": 0.1378591074787549,
  "maxInitializedRotationChangeDegrees": 26.175511324271802,
  "provisionalImuPositionPathAfterAppTriggerMetres": 3.982300124366205,
  "distanceReady": false
}


### Executed established-VIO feasibility - 7 October 2026

**Current status:** the offline comparison phase is complete. Do not request a
repeat on the unchanged APK or enable clinical distance stopping. Next work is
capture/calibration modernization and a startup convergence/readiness contract,
then native integration feasibility and independent physical validation.

Built pinned VINS-Mono estimator sources unchanged at
`90dabb5ec79946ae42fd2e1e91d4e69aabe1e25d`, with Ceres 1.14.0 at
`facb199f3eda902360f9e1d5271372b7e54febe1`, in Ubuntu 26.04 WSL2.
The reusable offline wrapper excludes ROS transport/visualization/loop closure,
uses static estimator storage matching upstream, and adds logging/header shims.
Its diagnostic Python LK/forward-backward/calibrated-epipolar frontend is not the
original VINS frontend. Five lens coefficients and native pixel rows are retained.
Hardware frame clocks include exposure/readout midpoint; the measured clock offset
is applied once by VINS. Acceleration is interpolated only at real gyro timestamps
between valid bracketing samples. No camera samples, steps or distance are invented.

Also compiled OpenVINS's ROS-disabled library at
`69488123ed9362dd44b6f28e7f4680abbff1442b`, with a documented build-only patch for
Boost 1.90's header-only system component. This is desktop build feasibility, not
Android/front-camera integration, rolling-shutter acceptance or a phone replay.
Dependency source and generated binaries remain local outside source commits.

The corrected harness passes an analytic non-gait camera/IMU control: 559 initialized
frames, rigid-only position RMSE 0.0003602 m. No similarity/scale alignment was fitted.
This verifies basic harness consistency under known geometry, not physical accuracy.
All 26 Python research checks pass, including five new timestamp/unit/profile/axis/reset
adapter checks. Themed notebook execution counts 36-40 retain inputs, preliminary
and corrected replay evidence, source/binary hashes and explicit assumptions.

| Actual retained capture | Initialized states after app movement trigger | Final state / resets | Diagnostic trajectory |
| --- | ---: | --- | --- |
| `1791347211905` | 154 | Initialized / 0 resets | Provisional IMU-position path 3.122 m, endpoint displacement 2.795 m; user originally reported a 3 m route. Not independent accuracy validation. |
| `1791349028491` | 161 | Initialized / 0 resets | Provisional IMU-position path 3.982 m, displacement 3.666 m; route length not restated, so no route error calculated. |
| `1791349358636` | 99 | Lost / 1 reset | Diverges to 23.13 m/s in an initialized state. No route distance admitted or stitched across reset. |

All inputs include setup and lack accepted full calibration. Replay uses JPEGs,
not live native luma/scheduling. First capture uses the separately retained matching
profile offline; its original device run had no profile. Reference noise parameters
are disclosed EuRoC-example placeholders, and camera/IMU translation begins at zero
with online refinement. No measured route length was used to fit scale.

**Critical new evidence:** "initialized" is not sufficient readiness. Estimated
camera/IMU separation briefly reaches 8.107 m in the first recording and 0.659 m in
the second during startup. After the app movement trigger, those maxima are 0.129 m
and 0.139 m, but that later plausibility does not validate their calibration. All
three runs change the initial rotation by about 26 degrees; stable first/second
runs change the time offset from -29.73 ms to about -3.63/-4.29 ms. These consistent
adjustments flag the imported rotation/timing candidate for investigation, not
proof of a uniquely identified calibration fault. The third run retains an
implausible ~0.35 m lever arm during walking and loses tracking.

No new app code, runtime, APK, OTA or Git publication accompanies this execution.
`distanceReady`, `fullCalibrationReady` and `independentDistanceValidation` remain
false. Continuous fusion is supported as a development direction, but the failure
and unstable startup calibration prevent adoption as a clinical stopping rule.
Do not select the pleasing 3.122 m value as success while ignoring the failed run.
Use the fixed digital-tablet calibration workflow, measured IMU noise, actual
20-30 Hz input where supported, and readiness based on stable geometry/tracking
before another independent route evaluation. A blanket 20-30 Hz target is engineering
guidance, not a proven universal minimum: the first two low-rate replays do initialize.



### Native capture preparation and measured-noise contract - 7 October 2026

This batch changes acquisition, not the VIO estimator or clinical endpoints. Camera/sensor callbacks copy data to a bounded storage actor; JPEG/tracking work no longer runs on the acquisition handler. The target is 30 camera Hz but exported metadata reports actual clocks, loss/backlog and delays. Phone performance remains unmeasured until build 12 is exercised.

A separate five-minute stationary, all-three-sensor capture provides data for short-term Allan noise review. The processor requires stationary SI values, monotonic nearly uniform timestamps and three-minute minimum coverage. White-noise candidates require a slope near -1/2; no bias random walk is fabricated. Optical-window selection binds new processed calibration to the device and capture. Digital board remains supported.

The readiness adapter is a contract for a **future continuous native estimator**. Its two-second stability/vector/physical/support limits are provisional engineering criteria. Original retained replay CSVs lack full vector telemetry and their input calibration/noise is incomplete, so all must remain blocked. These are artifact-admission checks, not new walking trials or fresh VINS executions.

In [41]:
from pathlib import Path
import subprocess,sys,json,hashlib,csv
ROOT=Path.cwd();sys.path.insert(0,str(ROOT/'android/research'))
from imu_noise import process_noise,characterize
from vio_readiness import assess_readiness
import numpy as np
base=ROOT/'android/dist/vio-benchmark';result={'execution':'new-tests-and-contracts-not-new-phone-capture','checks':{}}
python='C:/Users/frank/.venv/Scripts/python.exe'
commands={
 'python-research':[python,'-m','unittest','discover','-s','android/research','-p','test_*.py','-v'],
 'typescript':['npm.cmd','--prefix','android','run','ts:check'],
 'app-tests':['npm.cmd','--prefix','android','test'],
 'native-queue-compile':[str(ROOT/'android/dist/native-distance-tools/jdk-17.0.20.1+1/bin/javac.exe'),'-d',str(ROOT/'android/dist/native-distance-tools/test-classes'),str(ROOT/'android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/CaptureQueueBudget.java'),str(ROOT/'android/research/native-tests/CaptureQueueBudgetCheck.java')],
 'native-queue-run':[str(ROOT/'android/dist/native-distance-tools/jdk-17.0.20.1+1/bin/java.exe'),'-cp',str(ROOT/'android/dist/native-distance-tools/test-classes'),'CaptureQueueBudgetCheck']}
for name,command in commands.items():
 run=subprocess.run(command,cwd=ROOT,encoding='utf-8',errors='replace',capture_output=True)
 result['checks'][name]={'exitCode':run.returncode,'stdout':run.stdout,'stderr':run.stderr}
 if run.returncode:raise RuntimeError(name+' failed: '+run.stdout+run.stderr)
rng=np.random.default_rng(401);times=np.arange(40000,dtype=np.int64)*5_000_000
values=rng.normal(0,.0001/np.sqrt(.005),(len(times),3))
noise=characterize(times,values,'gyroscope')
result['analyticWhiteNoiseControl']={k:noise[k] for k in ['whiteNoiseSlopes','whiteNoiseDensityCandidate','whiteNoiseCandidateReady','biasRandomWalkReady']}
result['retainedCaptureNoiseAdmission']={}
result['retainedReplayReadiness']={}
for trial in sorted(base.glob('distance-trial-*')):
 if not trial.is_dir():continue
 archive=Path('C:/Users/frank/Downloads/Telegram Desktop')/(trial.name+'.zip')
 if archive.exists():result['retainedCaptureNoiseAdmission'][trial.name]=process_noise(archive)
 states=trial/'verified-states.csv'
 if states.exists():
  with states.open() as f:rows=list(csv.DictReader(f))
  result['retainedReplayReadiness'][trial.name]=assess_readiness(rows,{})
files=['android/research/imu_noise.py','android/research/vio_readiness.py','android/research/camera_calibration.py','android/src/screens/CameraCalibrationScreen.tsx','android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/GaitCameraCalibrationModule.kt','android/modules/gait-camera-calibration/android/src/main/java/expo/modules/gaitcalibration/CaptureQueueBudget.java']
result['sourceSha256']={name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in files}
(base/'capture-v2-validation.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps({**result,'checks':{k:{'exitCode':r['exitCode'],'outputTail':(r['stdout']+r['stderr'])[-550:]} for k,r in result['checks'].items()}},indent=2))

{
  "execution": "new-tests-and-contracts-not-new-phone-capture",
  "checks": {
    "python-research": {
      "exitCode": 0,
      "outputTail": "est_camera_gap_does_not_reset_map_and_continue_distance (test_visual_tracking.TrackingTests.test_camera_gap_does_not_reset_map_and_continue_distance) ... ok\ntest_one_map_preserves_relative_translation_scale (test_visual_tracking.TrackingTests.test_one_map_preserves_relative_translation_scale) ... ok\ntest_pure_rotation_does_not_establish_distance_scale (test_visual_tracking.TrackingTests.test_pure_rotation_does_not_establish_distance_scale) ... ok\n\n----------------------------------------------------------------------\nRan 34 tests in 5.239s\n\nOK\n"
    },
    "typescript": {
      "exitCode": 0,
      "outputTail": "\n> gait-steps@0.3.5 ts:check\n> tsc --noEmit\n\n"
    },
    "app-tests": {
      "exitCode": 0,
      "outputTail": "ot history (2.7123ms)\n\u2714 Observer ratings survive JSON review import and CSV without inventing an in

In [42]:
from pathlib import Path
import json,hashlib,subprocess
ROOT=Path.cwd();base=ROOT/'android/dist/vio-benchmark'
result=json.loads((base/'capture-v2-validation.json').read_text())
run=subprocess.run(['C:/Users/frank/.venv/Scripts/python.exe','-m','unittest','discover','-s','android/research','-p','test_*.py','-v'],cwd=ROOT,encoding='utf-8',errors='replace',capture_output=True)
if run.returncode:raise RuntimeError(run.stderr)
result['checks']['python-research']={'exitCode':run.returncode,'stdout':run.stdout,'stderr':run.stderr}
result['sourceSha256'].update({name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in result['sourceSha256']})
result['nativeBuildAttempts']={'12':{'id':'8ed55743-278d-4eab-8f69-bbf544dc4b8d','status':'failed','reason':'Kotlin JSONObject Map generic mismatch; corrected with explicit Map<*,*> cast'},'13':{'id':'478fc333-8bae-48b2-9666-47fa55196b74','status':'submitted'}}
result['runtimeFingerprint'] = json.loads((ROOT/'android/dist/native-distance-tools/build13-fingerprint.json').read_text())['hash']
(base/'capture-v2-validation.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps({'researchTestOutput':run.stderr[-500:],'sourceSha256':result['sourceSha256'],'nativeBuildAttempts':result['nativeBuildAttempts'],'runtimeFingerprint':result['runtimeFingerprint']},indent=2))

{
  "researchTestOutput": "tance (test_visual_tracking.TrackingTests.test_camera_gap_does_not_reset_map_and_continue_distance) ... ok\ntest_one_map_preserves_relative_translation_scale (test_visual_tracking.TrackingTests.test_one_map_preserves_relative_translation_scale) ... ok\ntest_pure_rotation_does_not_establish_distance_scale (test_visual_tracking.TrackingTests.test_pure_rotation_does_not_establish_distance_scale) ... ok\n\n----------------------------------------------------------------------\nRan 35 tests in 6.052s\n\nOK\n",
  "sourceSha256": {
    "android/research/imu_noise.py": "a33d333889ddfe68eaa1d2446d6c170b544f0ee0bcb847c804e0303986c18370",
    "android/research/vio_readiness.py": "87ab89f66cb12d9dbabc1504945efe5e993d360df9fa05730715b441d152ebec",
    "android/research/camera_calibration.py": "12220aed7215e4dc81487b67e8d9f8b5c12ecf03098a71e1c3de8e4df1721adb",
    "android/src/screens/CameraCalibrationScreen.tsx": "6f3c34d107dd9dc48c2ceb523183a3ba3c51a75bf647cb68499402c5f3

In [43]:
from pathlib import Path
import subprocess,json,hashlib
ROOT=Path.cwd();base=ROOT/'android/dist/vio-benchmark'
run=subprocess.run(['C:/Users/frank/.venv/Scripts/python.exe','-m','unittest','discover','-s','android/research','-p','test_*.py','-v'],cwd=ROOT,encoding='utf-8',errors='replace',capture_output=True)
if run.returncode:raise RuntimeError(run.stdout+run.stderr)
result=json.loads((base/'capture-v2-validation.json').read_text())
result['checks']['python-research']={'exitCode':0,'stdout':run.stdout,'stderr':run.stderr}
extra=['android/research/vio_reference.py','android/research/test_vio_reference.py','android/research/test_imu_noise.py','android/research/test_vio_readiness.py','android/research/native-tests/CaptureQueueBudgetCheck.java']
result['sourceSha256'].update({name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in list(result['sourceSha256'])+extra})
release=json.loads((ROOT/'android/dist/apk/GaitTrace-0.3.5-build13.release.json').read_text())
assert release['status']=='FINISHED' and release['nativeCaptureV2Methods'] and release['zipIntegrity']
assert release['runtimeVersion']==result['runtimeFingerprint']
result['nativeBuildAttempts']['13']={'id':release['buildId'],'status':'finished','apkVerified':True}
result['apkVerification']=release
result['physicalValidation']=False
(base/'capture-v2-validation.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps({'researchTestOutput':run.stderr[-500:],'nativeBuildAttempts':result['nativeBuildAttempts'],'apkVerification':release,'physicalValidation':False},indent=2))

{
  "researchTestOutput": "tance (test_visual_tracking.TrackingTests.test_camera_gap_does_not_reset_map_and_continue_distance) ... ok\ntest_one_map_preserves_relative_translation_scale (test_visual_tracking.TrackingTests.test_one_map_preserves_relative_translation_scale) ... ok\ntest_pure_rotation_does_not_establish_distance_scale (test_visual_tracking.TrackingTests.test_pure_rotation_does_not_establish_distance_scale) ... ok\n\n----------------------------------------------------------------------\nRan 35 tests in 6.202s\n\nOK\n",
  "nativeBuildAttempts": {
    "12": {
      "id": "8ed55743-278d-4eab-8f69-bbf544dc4b8d",
      "status": "failed",
      "reason": "Kotlin JSONObject Map generic mismatch; corrected with explicit Map<*,*> cast"
    },
    "13": {
      "id": "478fc333-8bae-48b2-9666-47fa55196b74",
      "status": "finished",
      "apkVerified": true
    }
  },
  "apkVerification": {
    "apk": "android\\dist\\apk\\GaitTrace-0.3.5-build13.apk",
    "bytes": 233341562,
    

### Final native build verification - 7 October 2026

0.3.5/build 13 completed and its downloaded APK is verified. Build 12 was a failed compiler attempt, not a release. The runtime fingerprint matches local source, EAS and the actual APK asset. New noise/optics/queue methods, package/version, ZIP integrity, permission boundaries, OpenCV ABIs and licenses are verified. Final 35 research checks pass, along with previously executed 177 app checks, TypeScript, real JVM queue checks and 10 browser flows. These checks do not prove phone throughput, audible guidance or metric distance.

Next: one device-only stationary noise capture and a new fixed digital-board capture through the usual holder. Export the two distinct ZIPs for analysis. The short noise capture does not complete long-term bias calibration. Full geometry/noise and continuous native VIO integration are still required before independent walking-distance validation. Native changes require installing build 13. No source push or OTA publication.

In [44]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "failed",
  "error": "Command '['wsl.exe', '-d', 'Ubuntu', '-u', 'root', '--', 'docker', 'run', '--rm', '--network', 'none', '--shm-size', '512m', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/dist/calibration-processor/synthetic-control,dst=/data', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/research/calibration-companion,dst=/bridge,readonly', 'gaittrace-ikalibr:reference-v1']' returned non-zero exit status 1.",
  "logTail": "/opt/ros/noetic/etc/catkin/profile.d/10.roslaunch.sh: line 3: ROS_MASTER_URI: unbound variable\n",
  "elapsedSeconds": 7.280999999959022,
  "sourceSha256": "373cca8b3cc090983939c3cb1a2cdb805f217d613b24985176c6d38660cd1983",
  "expected": {
    "source": "synthetic-known-answer-not-phone-or-gait-data",
    "T_camera_in_imu_m": [
      0.025,
      0.018,
      0.012
    ],
    "rotation_camera_in_imu_xyzw": [
      0,
     

In [45]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "failed",
  "error": "Command '['wsl.exe', '-d', 'Ubuntu', '-u', 'root', '--', 'docker', 'run', '--rm', '--network', 'none', '--shm-size', '512m', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/dist/calibration-processor/synthetic-control,dst=/data', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/research/calibration-companion,dst=/bridge,readonly', 'gaittrace-ikalibr:reference-v1']' returned non-zero exit status 1.",
  "logTail": "Traceback (most recent call last):\n  File \"/bridge/solve.py\", line 38, in <module>\n    if len(bases)!=1:raise ValueError('Reference initialization could not track this scene. The capture is kept.')\nValueError: Reference initialization could not track this scene. The capture is kept.\n",
  "elapsedSeconds": 17.640000000130385,
  "sourceSha256": "373cca8b3cc090983939c3cb1a2cdb805f217d613b24985176c6d38660cd1983",
  "expe

In [46]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "failed",
  "error": "Command '['wsl.exe', '-d', 'Ubuntu', '-u', 'root', '--', 'docker', 'run', '--rm', '--network', 'none', '--shm-size', '512m', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/dist/calibration-processor/synthetic-control,dst=/data', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/research/calibration-companion,dst=/bridge,readonly', 'gaittrace-ikalibr:reference-v1']' returned non-zero exit status 143.",
  "logTail": "",
  "elapsedSeconds": 359.1400000001304,
  "sourceSha256": "373cca8b3cc090983939c3cb1a2cdb805f217d613b24985176c6d38660cd1983",
  "expected": {
    "source": "synthetic-known-answer-not-phone-or-gait-data",
    "T_camera_in_imu_m": [
      0.025,
      0.018,
      0.012
    ],
    "rotation_camera_in_imu_xyzw": [
      0,
      0,
      0,
      1
    ],
    "time_offset_s": 0.0,
    "readout_s": 0.0,
    "frameRateHz":

In [47]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "failed",
  "error": "Command '['wsl.exe', '-d', 'Ubuntu', '-u', 'root', '--', 'docker', 'run', '--rm', '--network', 'none', '--shm-size', '512m', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/dist/calibration-processor/synthetic-control,dst=/data', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/research/calibration-companion,dst=/bridge,readonly', 'gaittrace-ikalibr:reference-v1']' returned non-zero exit status 143.",
  "logTail": "",
  "elapsedSeconds": 94.46900000004098,
  "sourceSha256": "373cca8b3cc090983939c3cb1a2cdb805f217d613b24985176c6d38660cd1983",
  "expected": {
    "source": "synthetic-known-answer-not-phone-or-gait-data",
    "T_camera_in_imu_m": [
      0.025,
      0.018,
      0.012
    ],
    "rotation_camera_in_imu_xyzw": [
      0,
      0,
      0,
      1
    ],
    "time_offset_s": 0.0,
    "readout_s": 0.0,
    "frameRateHz":

In [48]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "failed",
  "error": "Command '['wsl.exe', '-d', 'Ubuntu', '-u', 'root', '--', 'docker', 'run', '--rm', '--network', 'none', '--shm-size', '512m', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/dist/calibration-processor/synthetic-control,dst=/data', '--mount', 'type=bind,src=/mnt/c/Users/frank/Documents/MR-ICT Review Paper/android/research/calibration-companion,dst=/bridge,readonly', 'gaittrace-ikalibr:reference-v1']' returned non-zero exit status 1.",
  "logTail": "Traceback (most recent call last):\n  File \"/bridge/solve.py\", line 51, in <module>\n    cpu(['colmap','mapper','--database_path',str(db),'--image_path',str(images),'--output_path',str(sfm),'--Mapper.num_threads','2','--Mapper.init_min_tri_angle','25','--Mapper.init_max_error','2','--Mapper.tri_min_angle','3','--Mapper.ba_refine_focal_length','0','--Mapper.ba_refine_principal_point','0'],'sfm-map')\n  File \"/bridge/solv

In [49]:
# Published reference executable integration: fresh run on a synthetic control.
# This is not participant data, physical distance validation, or model training.
import sys,json,hashlib,time
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
folder=Path('android/dist/calibration-processor/synthetic-control').resolve()
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-control",
  "status": "completed",
  "software": {
    "image": "ulong2/ie_kalibr_image@sha256:2de8ac994a86951b6255e4b8485eae9982ea3ed03bac33dd9d0ee254e383587c",
    "executableSha256": "dc510fe4a9f84bac5bd258ea0b75445f3e246b4acda875c6c4800c2c0ffb17de"
  },
  "parameters": {
    "CalibParam": {
      "Header": {
        "Software": "iKalibr",
        "Version": "2.0.0",
        "Address": "https://github.com/Unsigned-Long/iKalibr.git"
      },
      "EXTRI": {
        "SO3_BiToBr": [
          {
            "key": "/gaittrace/imu",
            "value": {
              "qx": 0.0,
              "qy": 0.0,
              "qz": 0.0,
              "qw": 1.0
            }
          }
        ],
        "POS_BiInBr": [
          {
            "key": "/gaittrace/imu",
            "value": {
              "r0c0": 0.0,
              "r1c0": 0.0,
              "r2c0": 0.0
            }
          }
        ],
        "SO3_RjToBr": [],
        "POS_RjInBr"

In [50]:
# Analyse the freshly completed reference control; explicitly an artifact analysis.
# No participant accuracy or new clinical validation is implied.
import json,hashlib,math
from pathlib import Path
folder=Path('android/dist/calibration-processor/synthetic-control')
control=json.loads((folder/'control-outcome.json').read_text())
assert control['status']=='completed'
assert control['sourceSha256']==hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
p=control['parameters']['CalibParam'];e=control['expected']
position=p['EXTRI']['POS_CmInBr'][0]['value'];actual=[position[f'r{i}c0'] for i in range(3)]
translation_error=math.sqrt(sum((a-b)**2 for a,b in zip(actual,e['T_camera_in_imu_m'])))
q=p['EXTRI']['SO3_CmToBr'][0]['value'];rotation_error=math.degrees(2*math.acos(min(1,abs(q['qw']))))
offset=p['TEMPORAL']['TO_CmToBr'][0]['value'];readout=p['TEMPORAL']['RS_READOUT'][0]['value']
assert translation_error<0.01 and rotation_error<0.1 and abs(offset)<0.001 and abs(readout)<0.001
release=json.loads(Path('android/dist/native-distance-tools/build15-verification.json').read_text())
result={'execution':'artifact-analysis-of-fresh-reference-control','sourceSha256':control['sourceSha256'],'referenceSeconds':control['elapsedSeconds'],'translationErrorM':translation_error,'rotationErrorDegrees':rotation_error,'timeOffsetErrorS':abs(offset),'readoutErrorS':abs(readout),'controlChecks':'passed','limits':'synthetic numerical control, not physical phone accuracy','releaseVerification':release,'localCompanionSmoke':json.loads(Path('android/dist/native-distance-tools/companion-smoke.json').read_text()),'distanceReady':False,'physicalPhoneValidation':False}
(folder/'control-analysis.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "artifact-analysis-of-fresh-reference-control",
  "sourceSha256": "373cca8b3cc090983939c3cb1a2cdb805f217d613b24985176c6d38660cd1983",
  "referenceSeconds": 582.2029999999795,
  "translationErrorM": 0.005675609718087472,
  "rotationErrorDegrees": 0.016507724393377485,
  "timeOffsetErrorS": 9.89778957557244e-06,
  "readoutErrorS": 3.4040268871901156e-05,
  "controlChecks": "passed",
  "limits": "synthetic numerical control, not physical phone accuracy",
  "releaseVerification": {
    "apk": "android\\dist\\apk\\GaitTrace-0.3.6-build15.apk",
    "bytes": 233356422,
    "sha256": "592d865a76bc8e47f5ba747b25f4ffd3475cbfdda035655d2dd1b1d4d9951bb1",
    "manifest": {
      "versionCode": 15,
      "versionName": "0.3.6",
      "compileSdkVersion": 35,
      "compileSdkVersionCodename": "15",
      "package": "com.gaitsteps.app",
      "platformBuildVersionCode": 35,
      "platformBuildVersionName": 15
    },
    "cameraPermission": true,
    "microphonePermission": false,
  

In [51]:
# Fresh CUDA SfM + pinned published camera/IMU reference control.
# Same synthetic input as the CPU control; a new workspace and a new engine run.
import sys,json,hashlib,time,shutil
from pathlib import Path
sys.path.insert(0,str(Path('android/research').resolve()))
from calibration_companion import run_reference
original=Path('android/dist/calibration-processor/synthetic-control')
folder=Path('android/dist/calibration-processor/synthetic-control-gpu').resolve()
folder.mkdir(exist_ok=True)
if (folder/'result.json').exists():raise RuntimeError('Fresh GPU control requires a fresh workspace.')
shutil.copyfile(original/'input.zip',folder/'input.zip')
shutil.copyfile(original/'input.expected.json',folder/'input.expected.json')
began=time.monotonic()
try:
    report=run_reference(folder)
    result={'execution':'fresh-published-reference-cuda-sfm-control','status':'completed','software':report['software'],'parameters':report['upstreamParameters'],'sfm':report['provenance']['sfm'],'distanceReady':report['distanceReady']}
except Exception as error:
    result={'execution':'fresh-published-reference-cuda-sfm-control','status':'failed','error':str(error),'logTail':(folder/'processor.log').read_text(errors='replace')[-7000:]}
result['elapsedSeconds']=time.monotonic()-began
result['sourceSha256']=hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
result['expected']=json.loads((folder/'input.expected.json').read_text())
(folder/'control-outcome.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "fresh-published-reference-cuda-sfm-control",
  "status": "completed",
  "software": {
    "image": "ulong2/ie_kalibr_image@sha256:2de8ac994a86951b6255e4b8485eae9982ea3ed03bac33dd9d0ee254e383587c",
    "executableSha256": "dc510fe4a9f84bac5bd258ea0b75445f3e246b4acda875c6c4800c2c0ffb17de"
  },
  "parameters": {
    "CalibParam": {
      "Header": {
        "Software": "iKalibr",
        "Version": "2.0.0",
        "Address": "https://github.com/Unsigned-Long/iKalibr.git"
      },
      "EXTRI": {
        "SO3_BiToBr": [
          {
            "key": "/gaittrace/imu",
            "value": {
              "qx": 0.0,
              "qy": 0.0,
              "qz": 0.0,
              "qw": 1.0
            }
          }
        ],
        "POS_BiInBr": [
          {
            "key": "/gaittrace/imu",
            "value": {
              "r0c0": 0.0,
              "r1c0": 0.0,
              "r2c0": 0.0
            }
          }
        ],
        "SO3_RjToBr": [],
        "PO

In [52]:
# Artifact analysis of the freshly executed GPU-assisted published control.
# Not another solver run and not physical phone/clinical validation.
import json,hashlib,math,re,subprocess
from pathlib import Path
folder=Path('android/dist/calibration-processor/synthetic-control-gpu')
control=json.loads((folder/'control-outcome.json').read_text());p=control['parameters']['CalibParam'];e=control['expected']
assert control['status']=='completed' and control['sfm']['backend']=='cuda'
assert not (folder/'gpu-fallback.json').exists()
assert control['sourceSha256']==hashlib.sha256((folder/'input.zip').read_bytes()).hexdigest()
position=p['EXTRI']['POS_CmInBr'][0]['value'];actual=[position[f'r{i}c0'] for i in range(3)]
translation=math.sqrt(sum((a-b)**2 for a,b in zip(actual,e['T_camera_in_imu_m'])))
q=p['EXTRI']['SO3_CmToBr'][0]['value'];rotation=math.degrees(2*math.acos(min(1,abs(q['qw']))))
offset=abs(p['TEMPORAL']['TO_CmToBr'][0]['value']);readout=abs(p['TEMPORAL']['RS_READOUT'][0]['value'])
assert translation<0.01 and rotation<0.1 and offset<0.001 and readout<0.001
feature_log=(folder/'sfm-cuda-features.log').read_text();match_log=(folder/'sfm-cuda-matches.log').read_text()
assert 'Creating SIFT GPU feature extractor' in feature_log and 'Creating SIFT GPU feature matcher' in match_log
registered=int(re.search(r'Number of images:\s*(\d+)',(folder/'output/images/gaittrace/camera/sfm_ws/images.txt').read_text())[1])
assert registered==199
tests=subprocess.run(['C:/Users/frank/.venv/Scripts/python.exe','-m','unittest','discover','-s','android/research','-p','test_*.py'],capture_output=True,text=True)
assert tests.returncode==0 and 'Ran 40 tests' in tests.stderr
fingerprint=json.loads(Path('android/dist/native-distance-tools/current-calibration-fingerprint.json').read_text())
assert fingerprint['hash']=='e1383287be615da4c75a7a9ce08b9260cfd2cfb3'
result={'execution':'artifact-analysis-of-fresh-cuda-sfm-reference-control','translationErrorM':translation,'rotationErrorDegrees':rotation,'clockOffsetErrorS':offset,'readoutErrorS':readout,'registeredImages':registered,'stageSeconds':control['sfm']['stageSeconds'],'overallSecondsIncludingPreparation':control['elapsedSeconds'],'software':control['software'],'sfmSoftware':{k:control['sfm'][k] for k in ['image','colmapVersion','executableSha256']},'sourceSha256':control['sourceSha256'],'runtimeProof':json.loads(Path('android/dist/native-distance-tools/gpu-runtime-proof.json').read_text()),'freshResearchTests':tests.stderr.strip(),'androidRuntimeUnchanged':fingerprint['hash'],'sourceHashes':{str(path):hashlib.sha256(path.read_bytes()).hexdigest() for path in [Path('android/research/calibration_companion.py'),Path('android/research/calibration-companion/solve.py'),Path('android/research/calibration-companion/sfm.py'),Path('android/research/calibration-companion/Dockerfile.cuda')]},'wholeJobSpeedupClaimed':False,'limits':'Different COLMAP versions and cold preparation prevent a controlled CPU/GPU total-time comparison. Numerical synthetic control only; not physical phone or distance validation. Registered-image count is a post-run artifact check.','distanceReady':False,'fullCalibrationReady':False,'physicalPhoneValidation':False}
(folder/'control-analysis.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "execution": "artifact-analysis-of-fresh-cuda-sfm-reference-control",
  "translationErrorM": 0.006458331156416957,
  "rotationErrorDegrees": 0.033426110973413345,
  "clockOffsetErrorS": 0.00016065092108975135,
  "readoutErrorS": 0.0,
  "registeredImages": 199,
  "stageSeconds": {
    "features": 23.920340588000045,
    "matches": 18.95861584499994,
    "map": 43.48117439400016,
    "convert": 0.9672732420001466
  },
  "overallSecondsIncludingPreparation": 938.8589999999385,
  "software": {
    "image": "ulong2/ie_kalibr_image@sha256:2de8ac994a86951b6255e4b8485eae9982ea3ed03bac33dd9d0ee254e383587c",
    "executableSha256": "dc510fe4a9f84bac5bd258ea0b75445f3e246b4acda875c6c4800c2c0ffb17de"
  },
  "sfmSoftware": {
    "image": "colmap/colmap@sha256:c1de51615dff33066cd305071226b6343c1168324ef3bd622f0825e55feb5efe",
    "colmapVersion": [
      "COLMAP 4.2.1 (Commit Unknown on Unknown with CUDA, std hash maps)",
      "Usage:",
      "  colmap [command] [options]"
    ],
    "executable